# Skin-Lesion Classification on HAM10000: CNN Baseline, Transfer Learning and a Critical Comparison with DiffMIC

**CM4126 Computer Vision, Coursework 2026/27** | Dataset: [HAM10000](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000) | Reviewed paper: *[DiffMIC: Dual-Guidance Diffusion Network for Medical Image Classification](https://link.springer.com/chapter/10.1007/978-3-031-43987-2_10)* (MICCAI 2023)

## Contents

1. [Introduction](#1-introduction)
2. [Research and Data Exploration](#2-research-and-data-exploration): paper review, dataset choice, exploration, splits, image quality, enhancement
3. [Baseline Model](#3-baseline-model): training framework and a CNN trained from scratch
4. [Solution Improvement](#4-solution-improvement): transfer learning (3 pretrained models), class imbalance, image enhancement, GAN augmentation, plus a hyper-parameter search and a seed study
5. [Evaluation, Visualisation and Comparison](#5-evaluation-visualisation-and-comparison): test-set results, Grad-CAM, comparison with the baseline and with the paper
6. [Conclusion and Future Directions](#6-conclusion-and-future-directions)
7. [References](#references) and [Appendix](#appendix)

## 1. Introduction

**Problem.** Skin cancer is one of the most common cancers, and melanoma is its most dangerous form: caught early it is usually curable, caught late it is often fatal. A classifier for dermatoscopic images could act as a second reader and help prioritise patients.

**Task.** Classify a dermatoscopic image into one of 7 diagnostic categories (for example naevus, melanoma or basal cell carcinoma) on a highly imbalanced dataset, HAM10000.

**Objectives:**

| # | Objective | Where |
|---|---|---|
| 1 | Select and critically review a high-quality paper | §2.1 |
| 2 | Justify and describe the dataset (size, splits, classes, image quality, standardisation, enhancement) | §2.2–§2.10 |
| 3 | Build a CNN **baseline** that follows good practice, with every setting justified | §3 |
| 4 | **Improve** it: transfer learning (three pretrained models), imbalance handling, image enhancement, GAN images, a hyper-parameter search and a seed study | §4 |
| 5 | Evaluate with suitable metrics, **visualise** the results (learning curves, confusion matrices, Grad-CAM) and **compare** them with the baseline and the paper | §5 |
| 6 | Keep everything **reproducible** | whole notebook |

**Principles I followed throughout:**

1. **No data leakage.** HAM10000 often has several photos of one lesion, so I split the data by lesion, not by image. The paper's random image split is reproduced once to measure what it changes (§4.8, §5.6).
2. **Decisions use the validation set only.** The imbalance strategy, enhancement, hyper-parameters, early stopping and checkpoints never look at the test set, which is used only in §3.3 and §5.
3. **Accuracy is not enough.** 67 % of the images are one class, so always answering "naevus" is already 67 % accurate. The main metrics are **macro-F1** (the same family as the paper's F1), **balanced accuracy**, **melanoma recall** and one-vs-rest **AUC**. Per class I report recall, precision and F1.

### How to run this notebook

Open the notebook in VS Code, follow the set-up steps in the first code cell (§1.1) and press **Run All**; there is a single configuration.

* **Data:** if HAM10000 is not in `data/` yet, the first cells download it from Kaggle.
* **Saved results:** resized images and trained models are kept in `outputs/` and reloaded only if their settings and data are unchanged. Deleting `outputs/` forces a full re-run, which takes about an hour on the reference machine.
* **Reference machine:** AMD Ryzen 7 7800X3D, 64 GB RAM, AMD Radeon RX 7900 XT (20 GB), Windows 11, Python 3.12.10, PyTorch 2.9.1 (ROCm 7.2.1); speed benchmark in Appendix B.
* **Reproducibility:** every random generator uses seed 42 and the GPU kernels are deterministic. Another machine or driver can still give slightly different numbers, within the run-to-run noise measured in §4.6.

### 1.1 Set-up and configuration

All settings live in one cell, and each value is explained where it is used.

A few engineering choices make the notebook faster without changing what the models learn: a workaround for a BatchNorm bug in the Windows ROCm build of PyTorch (Appendix A); decoding the resized images once, caching them on disk and keeping them on the GPU, with augmentation on the GPU as well, because the CPU data loader was the bottleneck (Appendix B); 16-bit mixed precision, which was roughly two to five times faster than 32-bit in the benchmark; and saving trained models so that they are reused when their settings have not changed.

In [ ]:
#Set-up: create a virtual environment with Python 3.12 (e.g. `python -m venv venv`) and select it as the notebook kernel.
#Then, BEFORE running this cell, install PyTorch and torchvision built for your hardware:
#  NVIDIA, Linux:   `pip install torch torchvision` (the default build already supports CUDA).
#  NVIDIA, Windows: the default build is CPU-only; use the CUDA install command from https://pytorch.org/get-started/locally/
#  AMD, Linux:      use the ROCm install command from https://pytorch.org/get-started/locally/
#  AMD, Windows:    use AMD's ROCm build of PyTorch (see AMD's ROCm documentation); this is the set-up used here (Appendix A).
#  macOS:           `pip install torch torchvision`; current releases use the GPU only on Apple-silicon Macs (MPS), not NVIDIA or AMD.
#ROCm builds expose the GPU through torch.cuda, so NVIDIA and AMD GPUs are found the same way. Without a GPU the notebook still runs, but training takes far longer.
%pip install -q numpy pandas matplotlib opencv-python pillow scikit-learn tqdm kagglehub

In [ ]:
import os, platform, copy, hashlib, json, math, random, sys, time, warnings
from functools import partial
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Windows + AMD only: skip a GPU-detection probe that fails when the folder path contains spaces.
# Has to happen before torch is imported (Appendix A).
if os.name == "nt" and "ROCM_SDK_TARGET_FAMILY" not in os.environ:
    try:
        from rocm_sdk import _dist_info
        if "custom" in _dist_info.AVAILABLE_TARGET_FAMILIES:
            os.environ["ROCM_SDK_TARGET_FAMILY"] = "custom"
    except Exception:
        pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import models as tvm

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, precision_recall_fscore_support)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from tqdm import tqdm as _tqdm
from IPython.display import display

tqdm = partial(_tqdm, file=sys.stdout, ncols=100, mininterval=1)      # plain-text progress bars, saved with the notebook

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30, "display.width", 140)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.autolayout": True})
print(f"Python {platform.python_version()} | torch {torch.__version__} | torchvision {torchvision.__version__} | "
      f"numpy {np.__version__} | pandas {pd.__version__} | opencv {cv2.__version__}")

In [ ]:
# Settings shared by all experiments
SEED = 42

# images
IMG_SIZE = 224                           # input size of the ImageNet models
IMAGENET_MEAN = (0.485, 0.456, 0.406)    # ImageNet statistics, expected by the pretrained weights
IMAGENET_STD = (0.229, 0.224, 0.225)
N_CLASSES = 7

# training
BATCH_SIZE = 64                          # throughput stops rising here on this GPU (Appendix B)
EPOCHS_CNN = 40                          # baseline; early stopping usually ends it sooner
EPOCHS_HEAD = 3                          # transfer learning stage 1: new output layer only
EPOCHS_FINETUNE = 25                     # transfer learning stage 2: whole network
PATIENCE = 5                             # early stopping, pretrained models
PATIENCE_CNN = 8                         # the baseline learns more slowly
N_SEARCH_TRIALS = 6
SEARCH_FT_EPOCHS = 3                     # kept short on purpose
GAN_STEPS = 15000                        # about 134 passes over the training images at batch 64; not tuned
N_BOOTSTRAP = 1000                       # resamples for the 95 % confidence intervals

# reported by the paper on HAM10000 (Table 1b)
PAPER = {"name": "DiffMIC (Yang et al., MICCAI 2023)", "accuracy": 0.906, "f1": 0.816}

In [ ]:
# Download HAM10000 from Kaggle into data/ unless it is already there
import shutil, kagglehub
DATA_DIR = Path("data")
if not (DATA_DIR / "HAM10000_metadata.csv").exists():
    kaggle_dir = Path(kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000"))
    DATA_DIR.mkdir(exist_ok=True)
    for name in ["HAM10000_metadata.csv", "hmnist_8_8_L.csv", "hmnist_8_8_RGB.csv", "hmnist_28_28_L.csv", "hmnist_28_28_RGB.csv"]:
        shutil.copy2(kaggle_dir / name, DATA_DIR / name)
    for name in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        shutil.copytree(kaggle_dir / name, DATA_DIR / name, dirs_exist_ok=True)
print("data jpg files:", len(list(DATA_DIR.rglob("*.jpg"))))

In [ ]:
DATA_DIR = Path("data")
OUT_DIR = Path("outputs/checkpoints")    # trained weights and training histories
CACHE_DIR = Path("outputs/cache")        # resized images (.npy)
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

META_CSV = DATA_DIR / "HAM10000_metadata.csv"
IMAGE_PATHS = {p.stem: p for p in DATA_DIR.rglob("*.jpg")}      # image_id -> file
print("metadata  :", META_CSV.resolve())
print("jpg files :", len(IMAGE_PATHS))

In [ ]:
def set_seed(seed):
    """Seed every random generator and make the GPU kernels deterministic."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


if torch.cuda.is_available():                      # AMD ROCm also shows up as "cuda"
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Mixed precision (Appendix B). float16 needs a GradScaler; bfloat16 does not and is safer for transformers, so ViT uses it.
AMP_FP16 = torch.float16 if device.type == "cuda" else None
AMP_BF16 = torch.bfloat16 if device.type == "cuda" else None

set_seed(SEED)
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

In [ ]:
# Windows + AMD only (Appendix A): the ROCm build sends BatchNorm to MIOpen, whose run-time kernel compile fails with
# "'type_traits' file not found" (https://github.com/ROCm/TheRock/issues/9181). PyTorch's own GPU kernel does the same maths.
def apply_rocm_windows_batchnorm_fix():
    if os.name != "nt" or getattr(torch.version, "hip", None) is None:
        return False
    original = F.batch_norm

    def native_batch_norm(input, running_mean, running_var, weight=None, bias=None, training=False, momentum=0.1, eps=1e-5):
        if not input.is_cuda:
            return original(input, running_mean, running_var, weight, bias, training, momentum, eps)
        if training and input.numel() // max(input.shape[1], 1) <= 1:
            raise ValueError(f"Expected more than 1 value per channel when training, got {tuple(input.shape)}")
        previous = torch.backends.cudnn.enabled         # PyTorch 2.9 chooses the backend from this flag
        torch.backends.cudnn.enabled = False
        try:
            return torch.batch_norm(input, weight, bias, running_mean, running_var, training, momentum, eps, False)
        finally:
            torch.backends.cudnn.enabled = previous

    F.batch_norm = native_batch_norm
    return True


print("BatchNorm workaround for Windows + ROCm active:", apply_rocm_windows_batchnorm_fix())

In [ ]:
# Numbered captions: table captions go above the table, figure captions below the figure.
from IPython.display import Markdown

_CAPTION_COUNTER = {"Figure": 0, "Table": 0}


def caption(kind, text):
    _CAPTION_COUNTER[kind] += 1
    display(Markdown(f"**{kind} {_CAPTION_COUNTER[kind]}.** {text}"))


def show_fig(text):
    plt.show()
    caption("Figure", text)


def show_table(df, text):
    caption("Table", text)
    display(df)

## 2. Research and Data Exploration

### 2.1 Paper review

**Paper.** Yang, Y., Fu, H., Aviles-Rivero, A. I., Schönlieb, C.-B. and Zhu, L. (2023). *DiffMIC: Dual-Guidance Diffusion Network for Medical Image Classification.* MICCAI 2023, LNCS, pp. 95–105. DOI 10.1007/978-3-031-43987-2_10.

**Why this paper, and how I judged its quality.** MICCAI is a leading peer-reviewed medical-imaging conference. HAM10000 is one of the paper's three benchmarks, so its numbers are comparable with mine, and the code is public. The baselines are strong: ResNet, ViT, Swin, PVT and long-tail methods such as LDAM and ProCo. I judged quality by novelty, baseline strength, ablations and implementation detail, then read the paper critically for flaws in the evaluation.

**Method.** DiffMIC uses a diffusion model (DDPM) as a classifier: it adds noise to the label vector and learns to remove it, conditioned on the image, with a ResNet18 image embedding. Dual-granularity Conditional Guidance (DCG) adds a *global* prior from a saliency map of the whole image and a *local* prior from six 32×32 salient patches combined by gated attention. Both priors shift the noisy label and condition the denoiser, and a Maximum-Mean-Discrepancy loss (weight 0.5) pulls the predicted noise for each prior towards Gaussian noise. At inference the label is denoised from pure noise in T = 250 steps on HAM10000.

**Results.** On a 70/30 split of HAM10000, DiffMIC reports accuracy 0.906 and F1 0.816, against 0.887 and 0.763 for the best baseline, ProCo. The module ablation uses only the authors' placenta dataset, where diffusion, DCG and MMD each add roughly 1–3 accuracy points.

**Critical evaluation.**
1. *Possible leakage.* HAM10000 has 10,015 images of far fewer lesions, and the paper splits 7:3 without saying whether images of one lesion stay together, so its scores may be inflated. This is why I split by lesion.
2. *No variance.* Single runs without confidence intervals, so the gains over ProCo (0.019 accuracy, 0.053 F1) may be noise.
3. *Narrow metrics.* Only accuracy and F1 (averaging not stated); no sensitivity, AUC or melanoma recall.
4. *Weak support for some claims.* The ablation is not repeated on HAM10000, and the noise-removal claim rests on a t-SNE plot.
5. *Cost.* 1,000 epochs on an RTX 3090, and no validation protocol is described.

**What I take from it.** I use the same ResNet18 encoder, 224 px centre crop, flip and rotation augmentation and Adam (as AdamW), and report accuracy and macro-F1 for comparison. The data are split by lesion, and the paper's random split is run once to measure the leakage. Re-implementing the diffusion pipeline was beyond my compute and time, so §5.6 compares my results with the reported numbers and discusses the gap.

### 2.2 Why HAM10000

| Reason | Evidence |
|---|---|
| **Public, expert-labelled, citable** | 10,015 dermatoscopic images in 7 diagnostic classes; more than half of the diagnoses are confirmed by histopathology (Tschandl et al., 2018). |
| **Widely used benchmark** | It is the training set of ISIC 2018 Task 3 (Codella et al., 2019) and is used by the reviewed paper and the methods it compares with (DANIL, contrastive learning, ProCo), so results can be compared with the literature. |
| **Clinically important** | Melanoma detection is one of the best-known medical-imaging tasks (Esteva et al., 2017). |
| **Interesting problems** | strong class imbalance (about 58 : 1); a domain very different from ImageNet, which makes transfer learning a real question; artefacts such as hair, rulers and vignettes that motivate enhancement; and several images per lesion, which makes a leakage-free split necessary. |
| **Right size for the coursework** | about 2.8 GB and 10,000 images: every experiment trains in minutes on one GPU, and the data are still large enough for stable results. |

### 2.3 Dataset description

The metadata file has one row per image. The next cells load it and summarise the images.

In [ ]:
CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]            # alphabetical; this is the label order
CLASS_IDX = {c: i for i, c in enumerate(CLASSES)}
MEL = CLASS_IDX["mel"]
CLASS_INFO = {                                                          # code: (full name, category)
    "akiec": ("Actinic keratoses / intra-epithelial carcinoma", "pre-malignant"),
    "bcc":   ("Basal cell carcinoma", "malignant"),
    "bkl":   ("Benign keratosis-like lesions", "benign"),
    "df":    ("Dermatofibroma", "benign"),
    "mel":   ("Melanoma", "malignant"),
    "nv":    ("Melanocytic naevi", "benign"),
    "vasc":  ("Vascular lesions", "benign"),
}

meta = pd.read_csv(META_CSV)
meta["label"] = meta["dx"].map(CLASS_IDX)
meta["path"] = meta["image_id"].map(IMAGE_PATHS)
assert meta["path"].notna().all(), "some image files from the metadata were not found"

print(f"{len(meta)} images in the metadata")
show_table(meta.drop(columns=["path"]).head(), "First five rows of the metadata (one row per image).")

In [ ]:
# image sizes from the JPEG headers only, without decoding
def image_size(path):
    with Image.open(path) as im:
        return im.size                                   # (width, height)

with ThreadPoolExecutor(16) as pool:
    sizes = list(pool.map(image_size, meta["path"]))
size_counts = pd.Series(sizes).value_counts()

overview = pd.DataFrame({"value": {
    "images": len(meta),
    "distinct lesions (lesion_id)": meta["lesion_id"].nunique(),
    "images per lesion (mean)": round(len(meta) / meta["lesion_id"].nunique(), 2),
    "classes": meta["dx"].nunique(),
    "image size(s) in pixels (w x h)": ", ".join(f"{w}x{h} ({n})" for (w, h), n in size_counts.items()),
    "colour channels": "3 (RGB JPEG)",
    "images with missing age": int(meta["age"].isna().sum()),
    "total size on disk (GB)": round(sum(p.stat().st_size for p in meta["path"]) / 1e9, 2),
}})
show_table(overview, "Dataset overview: images, lesions, image size, channels, missing metadata and size on disk.")

class_table = (meta.groupby("dx").agg(images=("image_id", "count"), lesions=("lesion_id", "nunique"))
               .reindex(CLASSES))
class_table.insert(0, "category", [CLASS_INFO[c][1] for c in CLASSES])
class_table.insert(0, "diagnosis", [CLASS_INFO[c][0] for c in CLASSES])
class_table["share %"] = (100 * class_table["images"] / class_table["images"].sum()).round(1)
class_table.index.name = "class"
show_table(class_table.style.format({"share %": "{:.1f}"}).set_properties(subset=["diagnosis", "category"], **{"text-align": "left"}),
           "Images and lesions per diagnosis (the class code is the row label), with the clinical category and the share of the dataset.")

**Findings.** All images have the same format, 600 × 450 RGB JPEG. The 10,015 images show 7,470 distinct lesions, so some lesions appear more than once (Figure 1, right).

### 2.4 Class distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

counts = class_table["images"]
bars = axes[0].bar(CLASSES, counts, color=["#d95f02" if CLASS_INFO[c][1] != "benign" else "#1b9e77" for c in CLASSES])
for b, c in zip(bars, counts):
    axes[0].text(b.get_x() + b.get_width() / 2, c, f"{c}\n{100 * c / counts.sum():.1f}%", ha="center", va="bottom", fontsize=8)
axes[0].set_title("Images per class (orange = malignant / pre-malignant)")
axes[0].set_ylabel("number of images"); axes[0].set_ylim(0, counts.max() * 1.18)

axes[1].bar(CLASSES, counts, color="#7570b3"); axes[1].set_yscale("log")
axes[1].set_title("Same data, logarithmic axis"); axes[1].set_ylabel("number of images (log)")

per_lesion = meta.groupby("lesion_id").size()
axes[2].hist(per_lesion, bins=np.arange(1, per_lesion.max() + 2) - 0.5, color="#66a61e")
axes[2].set_yscale("log"); axes[2].set_xlabel("images of the same lesion"); axes[2].set_ylabel("number of lesions (log)")
axes[2].set_title("Several images of one lesion are common")
for ax in axes[:2]:
    ax.set_xlabel("diagnosis (class code, see the class table above)")
show_fig("Class distribution: images per class (left; orange = malignant or pre-malignant), the same on a log axis (middle) and how many images each lesion has (right).")

ratio = counts.max() / counts.min()
multi = (per_lesion > 1).sum()
print(f"imbalance ratio (largest / smallest class): {ratio:.1f} : 1")
print(f"majority-class accuracy (always predict 'nv'): {100 * counts.max() / counts.sum():.1f} %")
print(f"lesions photographed more than once: {multi} of {len(per_lesion)}  "
      f"({100 * meta['lesion_id'].map(per_lesion).gt(1).mean():.1f} % of all images belong to such lesions)")

**Findings.** `nv` is two thirds of the data, while `df` and `vasc` are only 1.1 % and 1.4 % (58 : 1). So accuracy alone is misleading (§1), the splits are stratified, and §4.2 compares a weighted loss with balanced sampling. Melanoma is the second-largest class (about 11 %), and I report it separately because a missed melanoma is the costliest error. 1,956 of the 7,470 lesions have more than one image, which is why the data are split by lesion (§2.6).

### 2.5 Patient and lesion metadata

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))

axes[0, 0].hist(meta["age"].dropna(), bins=range(0, 95, 5), color="#4c72b0")
axes[0, 0].set_title(f"Age (years); {int(meta['age'].isna().sum())} missing"); axes[0, 0].set_xlabel("age")

sex = meta["sex"].fillna("unknown").value_counts()
axes[0, 1].bar(sex.index, sex.values, color="#55a868"); axes[0, 1].set_title("Sex")

loc = meta["localization"].value_counts()
axes[0, 2].barh(loc.index[::-1], loc.values[::-1], color="#c44e52"); axes[0, 2].set_title("Body location")

dxt = meta["dx_type"].value_counts()
axes[1, 0].bar(dxt.index, dxt.values, color="#8172b2"); axes[1, 0].set_title("How the diagnosis was confirmed")
axes[1, 0].tick_params(axis="x", rotation=20)

data = [meta.loc[meta["dx"] == c, "age"].dropna() for c in CLASSES]
axes[1, 1].boxplot(data, tick_labels=CLASSES, showfliers=False); axes[1, 1].set_title("Age by class")

tab = pd.crosstab(meta["dx"], meta["dx_type"]).reindex(CLASSES)
(tab.div(tab.sum(axis=1), axis=0) * 100).plot(kind="bar", stacked=True, ax=axes[1, 2], legend=True, width=0.8)
axes[1, 2].set_title("Confirmation type within each class (%)"); axes[1, 2].set_xlabel(""); axes[1, 2].tick_params(axis="x", rotation=0)
axes[1, 2].legend(fontsize=7)
show_fig("Metadata overview: age, sex, body location, how the diagnosis was confirmed, age per class and confirmation type per class.")

**Findings.** Like the paper, I treat this as pure image classification and do not feed the metadata to the models. Age differs by class (melanoma, carcinomas and keratoses are more common in older patients), which the models do not control for. Most lesions are on the back, lower extremity and trunk. Some diagnoses rest on follow-up or consensus rather than histopathology, so part of the labels are expert opinion. The 57 missing ages do not matter because age is not used.

### 2.6 Train / validation / test splits

**Every lesion goes to exactly one split.** `StratifiedGroupKFold` with `groups = lesion_id` and `y = diagnosis` keeps all images of a lesion together and the class proportions about equal; two successive splits give about **70 % train, 15 % validation and 15 % test**. Validation drives every decision and the test set is only used for reporting (§3.3, §5). For the leakage experiment (§4.8) I also build the paper's protocol, a random 70/30 split of *images*, holding out 10 % of its training part for validation.

In [ ]:
def lesion_level_split(df, seed):
    """70/15/15 split, stratified by diagnosis and grouped by lesion. Returns row positions."""
    outer = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)          # 1/7 of the data -> test
    trainval, test = next(outer.split(df, df["dx"], groups=df["lesion_id"]))
    tv = df.iloc[trainval]
    inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed)          # 1/6 of the rest -> validation
    tr, va = next(inner.split(tv, tv["dx"], groups=tv["lesion_id"]))
    return trainval[tr], trainval[va], test


def paper_style_split(df, seed):
    """The paper's protocol: random 70/30 split of images (lesions ignored); 10 % of train held out for validation."""
    idx = np.arange(len(df))
    train, test = train_test_split(idx, test_size=0.30, stratify=df["dx"], random_state=seed)
    train, val = train_test_split(train, test_size=0.10, stratify=df["dx"].iloc[train], random_state=seed)
    return train, val, test


SPLITS = {}
for name, fn in [("lesion", lesion_level_split), ("paper", paper_style_split)]:
    tr, va, te = fn(meta, SEED)
    SPLITS[name] = {"train": np.sort(tr), "val": np.sort(va), "test": np.sort(te)}

# no lesion may appear in two parts of the lesion-level split
L = SPLITS["lesion"]
lesions = {k: set(meta["lesion_id"].iloc[v]) for k, v in L.items()}
assert not (lesions["train"] & lesions["val"]) and not (lesions["train"] & lesions["test"]) and not (lesions["val"] & lesions["test"])
print("lesion-level split: no lesion is shared between train, validation and test.\n")

# number of paper-style test images that show a lesion also in its training part
P = SPLITS["paper"]
train_lesions = set(meta["lesion_id"].iloc[P["train"]])
leaked = meta["lesion_id"].iloc[P["test"]].isin(train_lesions)
print(f"paper-style split: {leaked.sum()} of {len(P['test'])} test images ({100 * leaked.mean():.1f} %) show a lesion "
      f"that is also in the training set.\n")

rows = []
for proto in SPLITS:
    for part, idx in SPLITS[proto].items():
        c = meta["dx"].iloc[idx].value_counts().reindex(CLASSES).fillna(0).astype(int)
        rows.append({"protocol": proto, "split": part, "images": len(idx), "lesions": meta["lesion_id"].iloc[idx].nunique(),
                     **{k: v for k, v in c.items()}})
split_table = pd.DataFrame(rows)
show_table(split_table, "Images, lesions and class counts in the train, validation and test part of both split protocols.")

In [ ]:
# class proportions per split for both protocols, and the share of leaked test images
fig, axes = plt.subplots(1, 3, figsize=(16, 3.9), gridspec_kw={"width_ratios": [2.3, 2.3, 1.1]})
for ax, proto in zip(axes[:2], ["lesion", "paper"]):
    props = pd.DataFrame({part: meta["dx"].iloc[idx].value_counts(normalize=True).reindex(CLASSES).fillna(0) * 100
                          for part, idx in SPLITS[proto].items()})
    props.plot(kind="bar", ax=ax, width=0.8)
    ax.set_title(f"Class proportions per split ({proto} protocol)"); ax.set_ylabel("% of the split")
    ax.set_xlabel("diagnosis (class code)"); ax.tick_params(axis="x", rotation=0)

leak_share = {}
for proto in SPLITS:                                        # test images whose lesion is also in the training part
    train_l = set(meta["lesion_id"].iloc[SPLITS[proto]["train"]])
    leak_share[proto] = 100 * meta["lesion_id"].iloc[SPLITS[proto]["test"]].isin(train_l).mean()
axes[2].bar(["lesion\nprotocol", "paper\nprotocol"], [leak_share["lesion"], leak_share["paper"]], color=["#1b9e77", "#d95f02"])
for k, proto in enumerate(("lesion", "paper")):
    axes[2].text(k, leak_share[proto] + 1, f"{leak_share[proto]:.1f} %", ha="center")
axes[2].set_ylim(0, 50); axes[2].set_ylabel("% of test images with a lesion\nthat is also in the training set"); axes[2].set_title("Leakage into the test set")
show_fig("Left and middle: class proportions in the train, validation and test part of both split protocols (both are stratified, so the bars are almost equal by design). "
         f"Right: share of the test images whose lesion also appears in the training part ({leak_share['lesion']:.0f} % for the lesion-level protocol, {leak_share['paper']:.0f} % for the paper-style protocol).")

**Findings.** Both protocols are stratified, so the class mix is the same in every part and the left and middle panels look alike by design. The right panel shows the real difference: no test image of the lesion-level split shows a lesion from the training set, against 33 % (992 of 3,005) for the paper-style split. These near-duplicates are the leakage studied in §4.8 and §5.6. Everything below uses the lesion-level split unless stated otherwise.

### 2.7 Loading, centre-cropping and caching the images

Pretrained networks expect 224 × 224 input, and the images are 4:3. Like the paper, I **centre-crop the largest square (450 × 450) and resize it to 224 × 224** with Lanczos filtering. This keeps the lesion's proportions, and lesions sit near the centre anyway. Decoding 10,015 JPEGs in every epoch would take longer than the training itself, so the images are decoded **once**, saved as a `uint8` array (1.5 GB) and kept on the GPU (Appendix B).

In [ ]:
def load_square_224(path, size=IMG_SIZE):
    """Centre-crop the largest square of a JPEG and resize it; returns an RGB uint8 array."""
    with Image.open(path) as im:
        im = im.convert("RGB")
        w, h = im.size
        s = min(w, h)
        left, top = (w - s) // 2, (h - s) // 2
        return np.asarray(im.crop((left, top, left + s, top + s)).resize((size, size), Image.LANCZOS), dtype=np.uint8)


def cached_array(tag, ids, build_fn):
    """Load the uint8 array [N, H, W, 3] from CACHE_DIR, or build it with build_fn and save it."""
    key = hashlib.md5("|".join(ids).encode()).hexdigest()[:8]
    file = CACHE_DIR / f"cache_{tag}_{IMG_SIZE}_{len(ids)}_{key}.npy"
    if file.exists():
        print(f"[{tag}] loaded from {file.name}")
        return np.load(file)
    t0 = time.perf_counter()
    arr = build_fn()
    np.save(file, arr)
    print(f"[{tag}] built {arr.shape} in {time.perf_counter() - t0:.0f} s and saved to {file.name}")
    return arr


def _build_raw():
    with ThreadPoolExecutor(16) as pool:                                    # PIL releases the GIL while decoding
        return np.stack(list(tqdm(pool.map(load_square_224, meta["path"]), total=len(meta), desc="decoding images")))


IDS = meta["image_id"].tolist()
X_RAW = cached_array("raw", IDS, _build_raw)
Y = meta["label"].to_numpy()
print("raw cache:", X_RAW.shape, X_RAW.dtype, f"{X_RAW.nbytes / 1e9:.2f} GB")

### 2.8 Looking at the data and assessing image quality

Following the Week 6 advice to look at the data before training: five random images per class after cropping and resizing.

In [ ]:
def show_gallery(x, labels, per_class=5, seed=SEED, title=""):
    rng = np.random.RandomState(seed)
    fig, axes = plt.subplots(len(CLASSES), per_class, figsize=(2.0 * per_class, 2.05 * len(CLASSES)))
    for r, c in enumerate(CLASSES):
        pick = rng.choice(np.flatnonzero(labels == CLASS_IDX[c]), size=per_class, replace=False)
        for k in range(per_class):
            ax = axes[r, k]; ax.axis("off")
            if k < len(pick):
                ax.imshow(x[pick[k]])
            if k == 0:
                ax.set_title(f"{c}: {CLASS_INFO[c][0][:30]}", fontsize=8, loc="left")
    fig.suptitle(title, y=1.0); show_fig(title or "Random images of every class")


show_gallery(X_RAW, Y, title="Five random images per class (224 x 224, centre-cropped)")

**Observations.** Many images contain **hair** over the lesion; some have a dark circular vignette or are slightly blurred, and colour and lighting vary between devices and clinics. `mel`, `nv` and `bkl` often look alike.

Five **quality measures** per image (224 px version) put numbers on this: brightness (mean grey level), contrast (grey-level standard deviation), sharpness (variance of the Laplacian), the share of **dark corner** pixels (vignette) and the share of **hair** pixels (black-hat filter plus an elongation test).

In [ ]:
_HAIR_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))          # black-hat kernel for thin dark structures


def hair_mask(img_rgb, threshold=20, min_length=30, max_thickness=4.0):
    """Hair = thin dark structures (black-hat filter) that are also long and thin.

    A plain black-hat threshold also fires on pigment-network lines and dots inside the lesion (about 3 % of all
    pixels, see the check after the hair-removal figure), so a connected component is kept only if it is at least
    min_length pixels long and at most max_thickness pixels thick on average (area / length).
    """
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    dark = (cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, _HAIR_KERNEL) > threshold).astype(np.uint8)
    _, comp, stats, _ = cv2.connectedComponentsWithStats(dark, connectivity=8)
    length = np.maximum(stats[:, cv2.CC_STAT_WIDTH], stats[:, cv2.CC_STAT_HEIGHT])
    keep = (length >= min_length) & (stats[:, cv2.CC_STAT_AREA] / np.maximum(length, 1) <= max_thickness)
    keep[0] = False                                                     # label 0 is the background
    return keep[comp].astype(np.uint8)


def quality_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    k = 40                                                                  # 40 x 40 corner squares
    corners = np.concatenate([gray[:k, :k].ravel(), gray[:k, -k:].ravel(), gray[-k:, :k].ravel(), gray[-k:, -k:].ravel()])
    return (gray.mean(), gray.std(), cv2.Laplacian(gray, cv2.CV_64F).var(), (corners < 25).mean(), hair_mask(img).mean())


quality = pd.DataFrame([quality_features(im) for im in tqdm(X_RAW, desc="quality")],
                       columns=["brightness", "contrast", "sharpness", "dark_corner_frac", "hair_frac"])
quality["dx"] = meta["dx"].values
q_summary = quality.groupby("dx").mean(numeric_only=True).reindex(CLASSES)
q_summary[["dark_corner_frac", "hair_frac"]] *= 100                                  # as % of the pixels
q_summary = q_summary.rename(columns={"dark_corner_frac": "dark corner %", "hair_frac": "hair %"})
q_summary.index.name = "class"
show_table(q_summary.round(2), "Mean image-quality measures per diagnosis (dark corners and hair as % of the pixels).")

fig, axes = plt.subplots(1, 5, figsize=(17, 3.2))
for ax, col in zip(axes, ["brightness", "contrast", "sharpness", "dark_corner_frac", "hair_frac"]):
    # sharpness spans several orders of magnitude, so it gets log-spaced bins
    bins = np.logspace(np.log10(max(quality[col].min(), 1e-3)), np.log10(quality[col].max()), 40) if col == "sharpness" else 40
    ax.hist(quality[col], bins=bins, color="#4c72b0"); ax.set_title(col)
    ax.set_xlabel(col + (" (log scale)" if col == "sharpness" else ""))
axes[2].set_xscale("log"); axes[0].set_ylabel("number of images")
show_fig("Distribution of the five image-quality measures over all images (sharpness on a log axis).")

In [ ]:
# the worst examples of each problem, then how common it is (rough thresholds)
def show_examples(order, title, n=6):
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.5))
    for ax, i in zip(axes, order[:n]):
        ax.imshow(X_RAW[i]); ax.axis("off"); ax.set_title(f"{meta['dx'][i]}", fontsize=8)
    fig.suptitle(title, y=1.02); show_fig(title)


show_examples(np.argsort(quality["sharpness"].values), "Blurriest images (lowest Laplacian variance)")
show_examples(np.argsort(-quality["hair_frac"].values), "Most hair (highest black-hat fraction)")
show_examples(np.argsort(-quality["dark_corner_frac"].values), "Strongest dark vignette")
for t in (0.005, 0.01, 0.03):
    print(f"share of images whose hair fraction exceeds {100 * t:.1f} %: {100 * (quality['hair_frac'] > t).mean():.1f} %")
print(f"share of images with a dark vignette (> 20 % dark corner pixels): {100 * (quality['dark_corner_frac'] > 0.2).mean():.1f} %")
print(f"share of images with very low sharpness (Laplacian variance < 20): {100 * (quality['sharpness'] < 20).mean():.1f} %")

**Quality assessment and decisions.**

* **Size and format** are identical for every image; none is rejected.
* **Sharpness, brightness and contrast** vary between clinics, but no image is unusable, so I **keep all images**. Removing hard cases would make the test set unrealistically easy, and the brightness and contrast jitter of the augmentation (§3.1) covers the variation.
* **Hair**: 14.1 % of the images have more than 0.5 % hair pixels. The measure is a proxy (long, thin dark structures) that can also catch thin dark lesion borders. Hair removal is tested in §2.9 and §4.3.
* **Dark vignettes** almost disappear after the centre crop (0.2 % of the images).
* **The quality measures differ between classes** (Table 5): mean contrast is 20-22 for `akiec`, `bcc` and `df` but 28-34 for `nv`, `mel` and `vasc`, and mean sharpness ranges from 109 (`vasc`) to 203 (`bcc`). A model could exploit this bias; Grad-CAM (§5.4) checks where it looks.

### 2.9 Image enhancement

Two classical techniques from Week 1, applied once and cached:

1. **Hair removal**, similar to *DullRazor* (Lee et al., 1997): a black-hat filter finds thin dark structures, an elongation test keeps the long thin ones, and the dilated mask is filled by **inpainting** (Telea, OpenCV).
2. **CLAHE** (Zuiderveld, 1994) on the LAB lightness channel: it raises *local* contrast without amplifying noise as global histogram equalisation (Lab 5) does, and leaves the colours unchanged.

In [ ]:
def remove_hair(img_rgb, threshold=20):
    """Black-hat hair mask, dilated, then inpainted. Returns (clean image, mask)."""
    mask = hair_mask(img_rgb, threshold)
    mask = cv2.dilate(mask, np.ones((3, 3), np.uint8))
    return cv2.inpaint(img_rgb, mask * 255, 3, cv2.INPAINT_TELEA), mask


def clahe_lightness(img_rgb, clip=2.0, tiles=8):
    """CLAHE on the L channel of LAB (a new CLAHE object per call, it is not thread-safe)."""
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tiles, tiles)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def enhance(img_rgb):
    return clahe_lightness(remove_hair(img_rgb)[0])


# visual check on the six hairiest images
hairy = np.argsort(-quality["hair_frac"].values)[:6]
fig, axes = plt.subplots(4, 6, figsize=(13, 9))
for k, i in enumerate(hairy):
    img = X_RAW[i]; clean, mask = remove_hair(img)
    for r, (im, ttl) in enumerate([(img, "original"), (mask * 255, "detected hair"), (clean, "hair removed"), (clahe_lightness(clean), "+ CLAHE")]):
        axes[r, k].imshow(im, cmap="gray" if r == 1 else None); axes[r, k].axis("off")
        if k == 0:
            axes[r, k].set_title(ttl, loc="left", fontsize=9)
show_fig("Hair removal and CLAHE on the six images with the most hair-like structures: original, detected hair, hair removed, hair removed plus CLAHE.")

# share of pixels flagged by a plain black-hat threshold and by the final (dilated) mask, on 400 random images
sample = np.random.RandomState(1).choice(len(X_RAW), 400, replace=False)
plain_share = np.mean([(cv2.morphologyEx(cv2.cvtColor(X_RAW[i], cv2.COLOR_RGB2GRAY), cv2.MORPH_BLACKHAT, _HAIR_KERNEL) > 20).mean() for i in sample])
mask_share = np.mean([cv2.dilate(hair_mask(X_RAW[i]), np.ones((3, 3), np.uint8)).mean() for i in sample])
print(f"share of pixels flagged on 400 random images: plain black-hat threshold {100 * plain_share:.1f} %, hair mask after the elongation test and dilation {100 * mask_share:.2f} %")

In [ ]:
def _build_enh():
    with ThreadPoolExecutor(16) as pool:                                    # OpenCV releases the GIL too
        return np.stack(list(tqdm(pool.map(enhance, X_RAW), total=len(X_RAW), desc="enhancing")))


X_ENH = cached_array("enh2", IDS, _build_enh)     # "enh2": built after the hair filter was made stricter
print("enhanced cache:", X_ENH.shape)

**Findings.** On very hairy images the mask follows the thick hairs and inpainting removes them, leaving some smeared patches. Thin or light hairs stay, and thin dark lesion borders can still be caught. A plain black-hat threshold flags 2.7 % of the pixels on 400 random images and also fires inside the lesion; the elongation test brings this down to 0.58 % after dilation (about 0.2 % before dilation, Table 5). Whether this helps the classifier is tested in §4.3.

### 2.10 Standardisation

Pixel values are scaled to 0-1 and normalised per channel with the **ImageNet** statistics that the pretrained weights expect. The baseline uses the same values, so every model gets identical input. The next cell compares them with the statistics of the training images.

In [ ]:
# per-channel mean and std of the training pixels, in chunks (one float64 copy would need about 8 GB)
tr_rows = SPLITS["lesion"]["train"]
s1, s2, n_px = np.zeros(3), np.zeros(3), 0
for a in range(0, len(tr_rows), 512):
    chunk = X_RAW[tr_rows[a:a + 512]].reshape(-1, 3).astype(np.float64) / 255.0
    s1 += chunk.sum(0); s2 += (chunk ** 2).sum(0); n_px += len(chunk)
ds_mean = s1 / n_px
ds_std = np.sqrt(s2 / n_px - ds_mean ** 2)
show_table(pd.DataFrame({"HAM10000 train mean": ds_mean, "ImageNet mean": IMAGENET_MEAN,
                      "HAM10000 train std": ds_std, "ImageNet std": IMAGENET_STD}, index=list("RGB")).round(3),
           "Per-channel mean and standard deviation of the training pixels compared with the ImageNet statistics.")

**Findings.** The training images are much *redder* than ImageNet (red mean 0.76 against 0.49), brighter and less varied. I still use the **ImageNet values**, because the pretrained first-layer filters were learned on inputs normalised this way, and all models get the same scaling. In ResNet18 and EfficientNet-B0 the batch-norm layers also adapt their statistics during fine-tuning. I did not test dataset-specific statistics. Augmentation and normalisation run **on the GPU** inside the loader (§3.1); validation and test images are only normalised.

#### Summary of the data-preparation decisions

| Step | Decision | Justification |
|---|---|---|
| Split | by lesion, stratified, 70/15/15 | no leakage, the same class mix in every part, a separate validation set |
| Spatial | centre crop 450 × 450, resized to 224 × 224 | as in the paper; keeps the aspect ratio; fits the pretrained models |
| Quality | keep all images | realistic test set; variation covered by augmentation |
| Enhancement | hair removal + CLAHE, tested in §4.3 | decided by the validation result |
| Standardisation | ÷255, then ImageNet mean/std | what the pretrained weights expect; the same for all models |
| Storage | `uint8` cache on the GPU | removes the CPU decoding bottleneck (Appendix B) |

## 3. Baseline Model

### 3.1 Training framework

All experiments in §3 and §4 share one small framework, so they differ **only** in the thing being tested:

| Component | What it does |
|---|---|
| `GpuLoader` | serves batches straight from the `uint8` image cache on the GPU; can draw all classes equally often and add extra (synthetic) training images |
| `gpu_augment` | random flips, rotation, zoom, shift, brightness and contrast for a whole batch on the GPU |
| `build_model` | the baseline CNN, or a pretrained torchvision model with a new 7-class output layer |
| `train_model` | staged training with AdamW, learning-rate scheduling, early stopping on validation **macro-F1**, best-checkpoint restore, saving and loading |
| `compute_metrics` | accuracy, balanced accuracy, macro-F1, AUC, per-class recall, precision, F1 and specificity, melanoma recall |

**Augmentation.** A dermatoscopic image has **no fixed orientation** (unlike a chest X-ray, Week 6), so flips and rotations keep the label. Small zooms and shifts imitate framing differences, and ±20 % brightness and contrast jitter imitates devices and lighting; I leave hue and saturation alone because colour is diagnostic. Only **training images** are augmented.

In [ ]:
def to_device_u8(arr):
    """uint8 [N, H, W, 3] array -> uint8 [N, 3, H, W] tensor on the device."""
    return torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous().to(device)


MEAN_T = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
STD_T = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)


def normalise(x01):
    return (x01 - MEAN_T) / STD_T


# augmentation strength; "full" is the default, §4.3b compares all three
AUG_PRESETS = {
    "full": dict(rotation=180, flips=True, zoom=0.10, shift=0.05, brightness=0.20, contrast=0.20),
    "mild": dict(rotation=15, flips=False, zoom=0.05, shift=0.03, brightness=0.10, contrast=0.10),   # cautious, like the X-ray example in Week 6
    "none": None,
}
DEFAULT_AUG = "full"             # §4.3b may change it for the later experiments


def gpu_augment(x, preset="full"):
    """Random augmentation of a whole batch on the GPU. x: float [B, 3, H, W] in [0, 1]."""
    cfg = AUG_PRESETS[preset]
    if cfg is None:
        return x
    b, dev = x.shape[0], x.device
    u = lambda: torch.rand(b, device=dev) * 2 - 1                        # one value in [-1, 1] per image
    angle = u() * math.radians(cfg["rotation"])
    scale = 1.0 + cfg["zoom"] * u()
    shift_x, shift_y = cfg["shift"] * u(), cfg["shift"] * u()
    cos, sin = angle.cos() * scale, angle.sin() * scale
    # rotation, zoom and shift as one affine matrix per image
    theta = torch.stack([torch.stack([cos, -sin, shift_x], 1), torch.stack([sin, cos, shift_y], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    if cfg["flips"]:
        x = torch.where(torch.rand(b, 1, 1, 1, device=dev) < 0.5, x.flip(3), x)      # horizontal
        x = torch.where(torch.rand(b, 1, 1, 1, device=dev) < 0.5, x.flip(2), x)      # vertical
    x = x * (1 + cfg["brightness"] * u().view(b, 1, 1, 1))
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    x = (x - mean) * (1 + cfg["contrast"] * u().view(b, 1, 1, 1)) + mean             # contrast around the image mean
    return x.clamp(0, 1)


class GpuLoader:
    """Batches from a uint8 image tensor that is already on the GPU.

    train=True shuffles, augments and drops the last incomplete batch (BatchNorm needs more than one sample).
    balanced=True draws every class equally often. extra=(xe, ye) adds synthetic training images.
    """

    def __init__(self, x, y, idx, batch_size, train=False, balanced=False, extra=None, aug="full"):
        self.x, self.y, self.bs, self.train, self.balanced, self.aug = x, y, batch_size, train, balanced, aug
        self.idx = torch.as_tensor(idx, device=x.device, dtype=torch.long)
        self.xe, self.ye = extra if extra is not None else (None, None)
        self.n_main = len(self.idx)
        self.n_extra = 0 if self.xe is None else len(self.ye)
        if balanced:                                           # weight of an image = 1 / size of its class
            labels = torch.cat([y[self.idx], self.ye]) if self.n_extra else y[self.idx]
            self.w = (1.0 / torch.bincount(labels, minlength=N_CLASSES).clamp(min=1).float())[labels]

    def __len__(self):
        n = self.n_main + self.n_extra
        return n // self.bs if self.train else math.ceil(n / self.bs)

    def _gather(self, order):
        # positions below n_main are real images, the rest are synthetic
        if not self.n_extra:
            sel = self.idx[order]
            return self.x[sel], self.y[sel]
        real = order < self.n_main
        sel, extra_sel = self.idx[order[real]], order[~real] - self.n_main
        return torch.cat([self.x[sel], self.xe[extra_sel]]), torch.cat([self.y[sel], self.ye[extra_sel]])

    def __iter__(self):
        n = self.n_main + self.n_extra
        if not self.train:
            order = torch.arange(n, device=self.x.device)
        elif self.balanced:
            order = torch.multinomial(self.w, n, replacement=True)
        else:
            order = torch.randperm(n, device=self.x.device)
        for b in range(len(self)):
            xb, yb = self._gather(order[b * self.bs:(b + 1) * self.bs])
            xb = xb.float().div_(255)
            if self.train:
                xb = gpu_augment(xb, self.aug)
            yield normalise(xb), yb


X_RAW_T, X_ENH_T = to_device_u8(X_RAW), to_device_u8(X_ENH)
Y_T = torch.as_tensor(Y, device=device, dtype=torch.long)
print(f"image tensors on {device}: raw {tuple(X_RAW_T.shape)}, enhanced {tuple(X_ENH_T.shape)}, "
      f"{(X_RAW_T.numel() + X_ENH_T.numel()) / 1e9:.2f} GB")

In [ ]:
# eight random augmentations of one melanoma image
set_seed(SEED)
i = int(np.flatnonzero(Y == MEL)[0])
batch = X_RAW_T[i:i + 1].repeat(8, 1, 1, 1).float() / 255
aug = gpu_augment(batch)
fig, axes = plt.subplots(1, 9, figsize=(17, 2.2))
axes[0].imshow(batch[0].permute(1, 2, 0).cpu()); axes[0].set_title("original", fontsize=8)
for k in range(8):
    axes[k + 1].imshow(aug[k].permute(1, 2, 0).cpu()); axes[k + 1].set_title(f"augmented {k + 1}", fontsize=8)
for ax in axes: ax.axis("off")
show_fig("One melanoma image and eight random augmentations of it (training-time augmentation).")

In [ ]:
def compute_metrics(y, prob):
    """All metrics of this notebook from the true labels [N] and the predicted probabilities [N, 7]."""
    prob = prob.astype(np.float64)
    prob = prob / prob.sum(1, keepdims=True)                   # roc_auc_score checks that rows sum to 1
    pred = prob.argmax(1)
    precision, recall, f1, _ = precision_recall_fscore_support(y, pred, labels=range(N_CLASSES), zero_division=0)
    cm = confusion_matrix(y, pred, labels=range(N_CLASSES))
    tp = np.diag(cm)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    tn = cm.sum() - tp - fp - fn
    specificity = tn / np.maximum(tn + fp, 1)                  # one-vs-rest
    return {"accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
            "auc_macro": roc_auc_score(y, prob, multi_class="ovr", average="macro", labels=list(range(N_CLASSES))),
            "mel_recall": recall[MEL], "mel_specificity": specificity[MEL],
            "recall": recall, "precision": precision, "f1": f1, "specificity": specificity}


def bootstrap_ci(y, pred, metric, n=N_BOOTSTRAP, seed=SEED):
    """95 % percentile bootstrap interval of metric(y_true, y_pred), resampling the test images."""
    rng = np.random.RandomState(seed)
    values = [metric(y[i], pred[i]) for i in (rng.randint(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(values, [2.5, 97.5])


def inverse_frequency_weights(labels):
    """Class weights N / (K * n_c), as in Lab 5."""
    counts = np.bincount(labels, minlength=N_CLASSES).clip(min=1)
    return torch.tensor(len(labels) / (N_CLASSES * counts), dtype=torch.float32, device=device)

In [ ]:
class SimpleCNN(nn.Module):
    """Baseline CNN: 5 x [conv 3x3, BatchNorm, ReLU, max-pool], global average pooling, dropout, linear."""

    def __init__(self, num_classes=N_CLASSES, channels=(32, 64, 128, 256, 256), dropout=0.3):
        super().__init__()
        layers, prev = [], 3
        for c in channels:
            layers += [nn.Conv2d(prev, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            prev = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(prev, num_classes))

    def forward(self, x):
        return self.head(self.features(x))


def build_model(arch, pretrained=True, num_classes=N_CLASSES):
    """The model with a new num_classes output layer; pretrained weights are torchvision's ImageNet-1k."""
    if arch == "simple_cnn":
        return SimpleCNN(num_classes)
    if arch == "resnet18":
        m = tvm.resnet18(weights=tvm.ResNet18_Weights.DEFAULT if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif arch == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.DEFAULT if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif arch == "vit_b_16":
        m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.DEFAULT if pretrained else None)
        m.heads.head = nn.Linear(m.heads.head.in_features, num_classes)
    else:
        raise ValueError(arch)
    return m


def head_module(model, arch):
    """The new classifier, which transfer learning trains first."""
    attribute = {"simple_cnn": "head", "resnet18": "fc", "efficientnet_b0": "classifier", "vit_b_16": "heads"}[arch]
    return getattr(model, attribute)


net = build_model("simple_cnn")
n_params = sum(p.numel() for p in net.parameters())
print(f"SimpleCNN: {n_params / 1e6:.2f} M parameters")
with torch.no_grad():                                                         # feature-map size after each block
    x = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
    for layer in net.features:
        x = layer(x)
        if isinstance(layer, nn.MaxPool2d):
            print(f"  after pooling block: {tuple(x.shape)}")
del net

In [ ]:
@torch.no_grad()
def predict(model, loader, amp_dtype=None):
    """Mean loss, probabilities and labels over a loader (eval mode, no augmentation)."""
    model.eval()
    probs, ys, loss_sum, n = [], [], 0.0, 0
    for xb, yb in loader:
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            logits = model(xb)
        logits = logits.float()
        loss_sum += F.cross_entropy(logits, yb, reduction="sum").item()
        n += len(yb)
        probs.append(logits.softmax(1).cpu())
        ys.append(yb.cpu())
    return loss_sum / n, torch.cat(probs).numpy(), torch.cat(ys).numpy()


def train_one_epoch(model, loader, criterion, optimizer, scaler, amp_dtype, clip=1.0):
    model.train()
    loss_sum, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            logits = model(xb)
        loss = criterion(logits.float(), yb)
        if scaler.is_enabled():                     # float16: scale the loss so small gradients don't underflow
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
        loss_sum += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        n += len(yb)
    return loss_sum / n, correct / n


def fit_stage(model, train_loader, val_loader, *, epochs, param_groups, weight_decay, criterion, amp_dtype,
              patience, stage_name, history, verbose=True):
    """One training stage: AdamW + ReduceLROnPlateau, early stopping on validation macro-F1, best epoch restored."""
    optimizer = torch.optim.AdamW(param_groups, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.3, patience=2)
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16 and device.type == "cuda"))
    best_f1, best_state, best_epoch, bad_epochs = -1.0, None, 0, 0
    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, amp_dtype)
        val_loss, prob, y = predict(model, val_loader, amp_dtype)
        m = compute_metrics(y, prob)
        scheduler.step(m["macro_f1"])
        history.append({"stage": stage_name, "epoch": len(history) + 1, "train_loss": train_loss, "train_acc": train_acc,
                        "val_loss": val_loss, "val_acc": m["accuracy"], "val_macro_f1": m["macro_f1"],
                        "val_bal_acc": m["balanced_accuracy"], "lr": optimizer.param_groups[0]["lr"],
                        "seconds": time.perf_counter() - t0})
        if verbose:
            print(f"  [{stage_name}] epoch {epoch:2d}/{epochs}  train loss {train_loss:.3f} acc {train_acc:.3f} | "
                  f"val loss {val_loss:.3f} acc {m['accuracy']:.3f} macro-F1 {m['macro_f1']:.3f} | "
                  f"lr {optimizer.param_groups[0]['lr']:.1e} | {time.perf_counter() - t0:.1f}s")
        if m["macro_f1"] > best_f1 + 1e-4:
            best_f1, best_epoch, bad_epochs = m["macro_f1"], epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                if verbose:
                    print(f"  [{stage_name}] early stop (no improvement for {patience} epochs); best epoch {best_epoch}")
                break
    model.load_state_dict(best_state)                # back to the best epoch
    return best_f1

In [ ]:
RUNS = {}          # experiment name -> model, history, settings, training minutes


def data_fingerprint(data):
    """Short id of the images and the training split, so that outdated saved models are retrained."""
    split = hashlib.md5(np.ascontiguousarray(data["idx"]["train"]).tobytes()).hexdigest()[:6]
    return f"{split}-{len(data['x'])}-{float(data['x'][:16].float().mean()):.3f}"


def train_model(name, arch, data, *, pretrained, stages, class_weights=None, balanced_sampler=False, amp_dtype=None,
                extra=None, seed=SEED, weight_decay=1e-4, batch_size=BATCH_SIZE, patience=PATIENCE, verbose=True, aug=None):
    """Train one experiment, or load it if a model with exactly these settings was saved before.

    stages: list of {"name", "epochs", "lr", "train": "head" | "all", "head_mult"}. With "all" the whole network
            is fine-tuned, the pretrained layers at lr and the new layer at lr * head_mult.
    data:   {"x": uint8 images, "y": labels, "idx": {"train", "val", "test"}}
    """
    aug = aug or DEFAULT_AUG
    extra_id = 0 if extra is None else f"{len(extra[1])}-{hashlib.md5(extra[0][:8].cpu().numpy().tobytes()).hexdigest()[:6]}"
    config = {"arch": arch, "pretrained": pretrained, "stages": stages, "class_weights": class_weights is not None,
              "balanced_sampler": balanced_sampler, "extra_images": extra_id, "seed": seed, "weight_decay": weight_decay,
              "batch_size": batch_size, "patience": patience, "amp": str(amp_dtype), "data": data_fingerprint(data)}
    if aug != "full":                                     # only stored when it differs from the default
        config["aug"] = aug
    signature = hashlib.md5(json.dumps(config, sort_keys=True, default=str).encode()).hexdigest()[:10]
    ckpt, hist_file = OUT_DIR / f"ckpt_{name}.pt", OUT_DIR / f"hist_{name}.json"

    set_seed(seed)                                        # the new layer's initial weights depend on it
    model = build_model(arch, pretrained).to(device)

    if ckpt.exists() and hist_file.exists():
        saved = json.loads(hist_file.read_text())
        if saved.get("signature") == signature:
            model.load_state_dict(torch.load(ckpt, map_location=device))
            RUNS[name] = {"model": model, "history": saved["history"], "config": saved["config"], "minutes": saved["minutes"],
                          "from_cache": True, "arch": arch, "amp": amp_dtype}
            print(f"[{name}] loaded saved model")
            return RUNS[name]
        print(f"[{name}] settings or data changed since it was saved -> retraining")

    t0 = time.perf_counter()
    train_loader = GpuLoader(data["x"], data["y"], data["idx"]["train"], batch_size, train=True, balanced=balanced_sampler, extra=extra, aug=aug)
    val_loader = GpuLoader(data["x"], data["y"], data["idx"]["val"], 256)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    history, head = [], head_module(model, arch)
    head_ids = {id(p) for p in head.parameters()}
    for st in stages:
        if st["train"] == "head":
            for p in model.parameters():
                p.requires_grad = id(p) in head_ids
            groups = [{"params": list(head.parameters()), "lr": st["lr"]}]
        else:
            for p in model.parameters():
                p.requires_grad = True
            backbone = [p for p in model.parameters() if id(p) not in head_ids]
            groups = [{"params": backbone, "lr": st["lr"]}, {"params": list(head.parameters()), "lr": st["lr"] * st.get("head_mult", 10)}]
        if verbose:
            n_trainable = sum(p.numel() for g in groups for p in g["params"])
            n_all = sum(p.numel() for p in model.parameters())
            print(f"[{name}] stage '{st['name']}': {n_trainable:,} of {n_all:,} parameters trainable ({100 * n_trainable / n_all:.2f} %), lr {st['lr']:g}")
        fit_stage(model, train_loader, val_loader, epochs=st["epochs"], param_groups=groups, weight_decay=weight_decay,
                  criterion=criterion, amp_dtype=amp_dtype, patience=patience, stage_name=st["name"], history=history, verbose=verbose)
    minutes = (time.perf_counter() - t0) / 60
    torch.save(model.state_dict(), ckpt)
    hist_file.write_text(json.dumps({"history": history, "config": config, "minutes": minutes, "signature": signature}))
    RUNS[name] = {"model": model, "history": history, "config": config, "minutes": minutes, "from_cache": False,
                  "arch": arch, "amp": amp_dtype}
    print(f"[{name}] done in {minutes:.1f} min")
    return RUNS[name]


PRED_CACHE = {}


def evaluate(name, data, split):
    """Metrics, probabilities, labels and loss of a trained run on 'train', 'val' or 'test' (computed once)."""
    key = (name, split, id(data))
    if key not in PRED_CACHE:
        run = RUNS[name]
        loss, prob, y = predict(run["model"], GpuLoader(data["x"], data["y"], data["idx"][split], 256), run["amp"])
        PRED_CACHE[key] = (compute_metrics(y, prob), prob, y, loss)
    return PRED_CACHE[key]


def summary_row(name, m):
    return {"model": name, "accuracy": m["accuracy"], "balanced_acc": m["balanced_accuracy"], "macro_f1": m["macro_f1"],
            "auc": m["auc_macro"], "mel_recall": m["mel_recall"], "mel_spec": m["mel_specificity"]}


def per_class_table(names, data, split="val", labels=None, metrics=("recall", "precision", "f1")):
    """Per-class metrics of several runs: one row per class, one column per (metric, run).
    data is one data dict for all runs, or a list with one per run."""
    datas = data if isinstance(data, list) else [data] * len(names)
    labels = labels or names
    table = pd.DataFrame({(metric, label): evaluate(n, d, split)[0][metric]
                          for metric in metrics for n, d, label in zip(names, datas, labels)}, index=CLASSES)
    if all(d["idx"] is datas[0]["idx"] for d in datas):            # same images for every run
        table[("images", "")] = np.bincount(evaluate(names[0], datas[0], split)[2], minlength=N_CLASSES)
    return table


def plot_histories(names, title="", figsize=(14, 4)):
    """Loss, validation macro-F1 and accuracy curves of several runs."""
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    for n in names:
        h = pd.DataFrame(RUNS[n]["history"])
        line, = axes[0].plot(h["epoch"], h["train_loss"], label=n)
        axes[0].plot(h["epoch"], h["val_loss"], "--", color=line.get_color())
        axes[1].plot(h["epoch"], h["val_macro_f1"], color=line.get_color(), label=n)
        axes[2].plot(h["epoch"], h["train_acc"], color=line.get_color(), label=f"{n} train")
        axes[2].plot(h["epoch"], h["val_acc"], "--", color=line.get_color())
    axes[0].set_title("loss (solid: train, dashed: validation)")
    axes[1].set_title("validation macro-F1")
    axes[2].set_title("accuracy (solid: train, dashed: validation)")
    for ax in axes:
        ax.set_xlabel("epoch")
    axes[0].legend(fontsize=7)
    fig.suptitle(title)
    show_fig(title or "Learning curves")


def plot_confusion(y, prob, title="", ax=None):
    """Row-normalised confusion matrix with counts (the diagonal is the recall of each class)."""
    cm = confusion_matrix(y, prob.argmax(1), labels=range(N_CLASSES))
    norm = cm / cm.sum(1, keepdims=True).clip(min=1)
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.grid(False)
    ax.set_xticks(range(N_CLASSES), CLASSES, rotation=45)
    ax.set_yticks(range(N_CLASSES), CLASSES)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{100 * norm[i, j]:.0f}%\n({cm[i, j]})", ha="center", va="center", fontsize=6.5,
                    color="white" if norm[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title, fontsize=9)
    if standalone:
        show_fig(title or "Row-normalised confusion matrix")


DATA = {"x": X_RAW_T, "y": Y_T, "idx": SPLITS["lesion"]}              # raw images, lesion-level split
DATA_ENH = {"x": X_ENH_T, "y": Y_T, "idx": SPLITS["lesion"]}          # enhanced images, same split

### 3.1b Sanity check: overfitting one tiny batch

Following the Week 6 tuning recipe, a fresh network is first trained on a handful of images with dropout and augmentation off, and the training loss must go close to zero. I use 14 images (2 per class) for the baseline CNN and for a randomly initialised ResNet18. The loss starts near ln 7 = 1.95 (1.93 and 2.08 below).

In [ ]:
def overfit_tiny_batch(arch, per_class=2, steps=80, lr=1e-3, seed=SEED):
    """Train a fresh network on a few images per class, dropout and augmentation off; returns the loss of every step."""
    set_seed(seed)
    tr = SPLITS["lesion"]["train"]
    pick = np.concatenate([tr[Y[tr] == c][:per_class] for c in range(N_CLASSES)])
    xb = normalise(X_RAW_T[torch.as_tensor(pick, device=device)].float() / 255)
    yb = Y_T[torch.as_tensor(pick, device=device)]
    model = build_model(arch, pretrained=False).to(device).train()
    for m in model.modules():                                                               # dropout would stop the loss reaching zero
        if isinstance(m, nn.Dropout):
            m.p = 0.0
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0)
    losses = []
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(xb), yb)
        loss.backward(); opt.step()
        losses.append(loss.item())
    return losses


sanity = {arch: overfit_tiny_batch(arch) for arch in ("simple_cnn", "resnet18")}
fig, ax = plt.subplots(figsize=(6, 3.4))
for arch, l in sanity.items():
    ax.plot(l, label=f"{arch}: {l[0]:.2f} -> {l[-1]:.3f}")
ax.set_yscale("log"); ax.set_xlabel("step"); ax.set_ylabel("training loss"); ax.legend()
show_fig("Sanity check: training loss of fresh networks trained on 14 images (2 per class) with dropout and augmentation off.")
for arch, l in sanity.items():
    print(f"{arch:11s}: loss {l[0]:.3f} -> {l[-1]:.4f}")

### 3.2 Baseline architecture and settings

The baseline is a small CNN trained **from scratch**, kept compact (about 1.0 M parameters) because a deep network trained from scratch converges slowly and can run out of memory.

| Setting | Choice | Justification |
|---|---|---|
| Blocks | 5 × [Conv 3×3 → BatchNorm → ReLU → MaxPool 2×2], channels 32-64-128-256-256 | the pattern of Weeks 3–4 (small filters, more channels, smaller maps); a 224 px input becomes a 7×7 map after 5 poolings |
| Batch-norm | after every convolution | faster, more stable training that allows a larger learning rate (Ioffe & Szegedy, 2015; Week 4) |
| Output | global average pooling → dropout 0.3 → linear(7) | far fewer parameters than a flatten + dense head, so less over-fitting; dropout as regulariser (Srivastava et al., 2014; Week 4) |
| Optimiser | AdamW, lr 1e-3, weight decay 1e-4 | Adam is the lecture default (Kingma & Ba, 2015) and 1e-3 the Week 6 starting point for a new network; AdamW decouples weight decay (Loshchilov & Hutter, 2019) |
| Batch size | 64 | the lectures suggest 16–64 and small batches generalise well (Bengio, 2012; Masters & Luschi, 2018); 64 is the top of that range and where throughput on this GPU stops rising (Appendix B); §4.4 also tries 32 and 128 |
| Loss | cross-entropy, plain *or* class-weighted (both tested) | multi-class classification; the weights counter the 58 : 1 imbalance |
| LR schedule | ReduceLROnPlateau (×0.3, patience 2) on validation macro-F1 | Week 6 default: lowers the rate only when progress stalls |
| Stopping | early stopping on validation macro-F1 (patience 8), best epoch restored | prevents over-fitting; macro-F1 accounts for the imbalance |
| Precision | float16 mixed precision with gradient scaling | faster training at the same accuracy (Micikevicius et al., 2018; Appendix B) |
| Augmentation | §3.1 | label-preserving for dermatoscopy |

I train two variants, with **plain** and with **class-weighted** cross-entropy, and the one with the better *validation* macro-F1 becomes *the* baseline.

In [ ]:
# baseline CNN with plain and with class-weighted cross-entropy
cnn_stage = [{"name": "scratch", "epochs": EPOCHS_CNN, "lr": 1e-3, "train": "all", "head_mult": 1}]
W_TRAIN = inverse_frequency_weights(Y[SPLITS["lesion"]["train"]])
print("class weights (inverse frequency):", {c: round(float(w), 2) for c, w in zip(CLASSES, W_TRAIN.cpu())})

train_model("cnn_plain", "simple_cnn", DATA, pretrained=False, stages=cnn_stage, amp_dtype=AMP_FP16, patience=PATIENCE_CNN)
_ = train_model("cnn_weighted", "simple_cnn", DATA, pretrained=False, stages=cnn_stage, class_weights=W_TRAIN, amp_dtype=AMP_FP16,
            patience=PATIENCE_CNN)

In [ ]:
# the variant with the better validation macro-F1 becomes the baseline
plot_histories(["cnn_plain", "cnn_weighted"], "Baseline CNN: plain vs class-weighted loss (validation used for monitoring)")
val_table = pd.DataFrame([summary_row(n, evaluate(n, DATA, "val")[0]) for n in ["cnn_plain", "cnn_weighted"]]).set_index("model")
show_table(val_table.round(3), "The two baseline variants on the validation set.")
BASELINE = val_table["macro_f1"].idxmax()
print("baseline (better validation macro-F1):", BASELINE)

### 3.3 Baseline results

The next cell evaluates the chosen baseline on the **training, validation and test** sets. In Figure 12 the validation curves of both variants follow the training curves; for the weighted variant validation accuracy is even above training accuracy, because training accuracy is measured on augmented batches with dropout on.

In [ ]:
# the chosen baseline on train / validation / test, then per-class results, confusion matrix and a confidence interval
rows = []
for split in ["train", "val", "test"]:
    m = evaluate(BASELINE, DATA, split)[0]
    rows.append({"split": split, **{k: m[k] for k in ["accuracy", "balanced_accuracy", "macro_f1", "auc_macro", "mel_recall", "mel_specificity"]}})
base_table = pd.DataFrame(rows).set_index("split")
show_table(base_table.round(3), "The chosen baseline on the training, validation and test set.")

m_test, prob_test, y_test, _ = evaluate(BASELINE, DATA, "test")
rec = pd.DataFrame({"recall (sensitivity)": m_test["recall"], "specificity": m_test["specificity"], "precision": m_test["precision"], "f1": m_test["f1"],
                    "test images": np.bincount(y_test, minlength=N_CLASSES)}, index=CLASSES)
show_table(rec.round(3), "Baseline per-class sensitivity (recall), specificity, precision and F1 on the test set.")
plot_confusion(y_test, prob_test, f"Baseline ({BASELINE}), test set")
lo, hi = bootstrap_ci(y_test, prob_test.argmax(1), lambda a, b: f1_score(a, b, average="macro", zero_division=0))
print(f"baseline test macro-F1 = {m_test['macro_f1']:.3f}  (95 % bootstrap CI {lo:.3f} to {hi:.3f})")

# reference: a "classifier" that always answers nv, the majority class
ref_prob = np.full_like(prob_test, 1e-6); ref_prob[:, CLASS_IDX["nv"]] = 1.0
m_ref = compute_metrics(y_test, ref_prob)
ref_table = pd.DataFrame({"always answers nv": m_ref, "baseline CNN": m_test}).loc[["accuracy", "balanced_accuracy", "macro_f1", "mel_recall"]]
show_table(ref_table.astype(float).round(3), "Test-set reference: a classifier that always answers nv (majority class) against the baseline CNN.")

**Summary of the baseline.**

* **Under-fitting, not over-fitting.** Macro-F1 is 0.482 / 0.479 / 0.492 on train / validation / test and accuracy 0.673 / 0.654 / 0.673 (Table 8): a 1 M-parameter network trained from scratch for at most 40 epochs on about 7,000 images has not even learned its training images well. Validation and test agree, so choosing the variant on validation did not over-fit.
* **Accuracy hides the gain.** Test accuracy (0.673) is no better than always answering `nv` (0.669), but macro-F1 is 0.492 against 0.115 and melanoma recall 0.484 against 0 (Table 10).
* **Where it fails.** Recall is 45-48 % for `akiec`, `bcc`, `bkl`, `df` and `mel`, 76 % for `nv` and 80 % for `vasc` (20 test images). The up-weighted rare classes are predicted too often, so precision is low (14 % for `df`). 102 of 958 naevi are called melanoma, and only 77 of 159 melanomas are found.
* **What §4 tries.** Pretrained networks have learned edges, textures and colours from 1.2 million ImageNet images; the baseline has to learn them from about 7,000. Class weighting is a cheap first fix, and §4.2 compares it with balanced sampling on a stronger model.

## 4. Solution Improvement

I improve the baseline one step at a time. **Each step changes one thing** and is judged on the **validation** set; no decision uses the test set. Every experiment also gets a per-class table (recall, precision and F1), because the overall scores hide what happens in the rare classes.

| Step | Experiment | Section |
|---|---|---|
| A | **Transfer learning**: two-stage fine-tuning of pretrained models (against the from-scratch baseline in §4.5) | 4.1 |
| B | **Class imbalance**: nothing, weighted loss or balanced sampling | 4.2 |
| C | **Hair removal + CLAHE** against raw images | 4.3 |
| C2 | **Flips and large rotations** in the augmentation | 4.3b |
| D | **Learning rate, weight decay, batch size** (random search) | 4.4 |
| E | **Other pretrained models**: EfficientNet-B0 and ViT-B/16 against ResNet18 | 4.5 |
| F | **Run-to-run noise** (3 seeds) | 4.6 |
| G | **GAN-generated images** of the rare classes | 4.7 |
| H | Effect of the paper's **leaky split** | 4.8 |

### 4.1 Transfer learning: design and justification

I compare three architectures of different design and size, all with ImageNet-1k weights from torchvision:

| Model | Parameters | Why it is included |
|---|---|---|
| **ResNet18** (He et al., 2016) | 11.2 M | the image encoder of the reviewed paper; small and fast, so the natural reference; residual connections keep training stable (Week 4) |
| **EfficientNet-B0** (Tan & Le, 2019) | 4.0 M | a modern CNN found by architecture search and compound scaling, with different building blocks (depth-wise convolutions, squeeze-and-excitation) |
| **ViT-B/16** (Dosovitskiy et al., 2021) | 85.8 M | a transformer (Week 5), to test whether attention-based features transfer better to dermatoscopy; the paper also compares with ViT and Swin |

**Two-stage strategy (Weeks 4 and 6).** *Stage 1:* the pretrained weights are frozen and only the new 7-class output layer is trained (3 epochs, lr 1e-3); the BatchNorm layers of ResNet18 and EfficientNet-B0 still update their running statistics. *Stage 2:* the **whole** network is fine-tuned with a **much smaller learning rate**, 10× higher for the new layer. The Week 6 decision table recommends fine-tuning most of the network for a moderate-sized dataset that differs from ImageNet. Raghu et al. (2019) found that transfer to medical images helps mainly through a better starting point and faster convergence, and Yosinski et al. (2014) showed that transferred features still help after fine-tuning.

In [ ]:
def tl_stages(ft_lr, head_epochs=None, ft_epochs=None):
    """Stage 1 trains only the new classifier, stage 2 fine-tunes everything with a small learning rate."""
    return [{"name": "head", "epochs": head_epochs or EPOCHS_HEAD, "lr": 1e-3, "train": "head"},
            {"name": "finetune", "epochs": ft_epochs or EPOCHS_FINETUNE, "lr": ft_lr, "train": "all", "head_mult": 10}]


FT_LR_DEFAULT = 1e-4          # the usual fine-tuning range is 1e-5 to 1e-4 (Week 6)

### 4.2 Class imbalance: weighted loss and balanced sampling

I try three options on the same pretrained ResNet18, with everything else unchanged:

1. **Plain** cross-entropy.
2. **Weighted** cross-entropy with class weights $N / (K\,n_c)$, as in Lab 5.
3. **Balanced sampling**: every class is drawn equally often, so rare images are repeated, each time with a different augmentation.

In [ ]:
# three ways of handling the imbalance; the best on validation is used from here on (one fix, not both)
IMB_OPTIONS = {"plain": {}, "weighted": {"class_weights": W_TRAIN}, "sampler": {"balanced_sampler": True}}
for opt, kw in IMB_OPTIONS.items():
    train_model(f"r18_{opt}", "resnet18", DATA, pretrained=True, stages=tl_stages(FT_LR_DEFAULT), amp_dtype=AMP_FP16, **kw)

imb_names = [f"r18_{o}" for o in IMB_OPTIONS]
plot_histories(imb_names, "ResNet18: class-imbalance strategies (validation)")
imb_table = pd.DataFrame([summary_row(n, evaluate(n, DATA, "val")[0]) for n in imb_names]).set_index("model")
show_table(imb_table.round(3), "ResNet18 with three class-imbalance strategies on the validation set.")
IMB_BEST = imb_table["macro_f1"].idxmax().replace("r18_", "")
print("best imbalance strategy on validation:", IMB_BEST)

In [ ]:
def plot_class_recall(names, data, split="val", title=""):
    fig, ax = plt.subplots(figsize=(10, 3.6))
    w = 0.8 / len(names)
    for k, n in enumerate(names):
        ax.bar(np.arange(N_CLASSES) + k * w, evaluate(n, data, split)[0]["recall"], w, label=n)
    ax.set_xticks(np.arange(N_CLASSES) + 0.4 - w / 2, CLASSES); ax.set_ylabel("recall"); ax.set_ylim(0, 1.05)
    ax.set_title(title or f"Per-class recall on the {split} set"); ax.legend(fontsize=7); show_fig(title or f"Per-class recall on the {split} set")


plot_class_recall(imb_names, DATA, "val", "ResNet18: per-class recall on the validation set")
show_table(per_class_table(imb_names, DATA, labels=list(IMB_OPTIONS)).round(3),
           "ResNet18 with three class-imbalance strategies: per-class recall, precision and F1 on the validation set.")

**Result.** Without any remedy, validation accuracy is 0.819, macro-F1 0.670 and melanoma recall 0.509. The weighted loss gives 0.774 / 0.719 / 0.642 and the balanced sampler 0.784 / 0.724 / 0.642. Both remedies give up 3.5-4.5 points of accuracy for about +0.05 macro-F1 and +0.13 melanoma recall. They end within 0.005 of each other, which is about the seed noise measured in §4.6, so it is a tie; I keep the sampler because it scored slightly higher. Per class (Table 12), both remedies raise the recall of `akiec`, `df`, `vasc` and `mel` and lower that of `nv` (0.911 to 0.810-0.825). The macro-F1 gain comes from `akiec`, `df` and `vasc` (sampler F1 0.648, 0.778 and 0.800 against 0.433, 0.643 and 0.703); melanoma F1 does not improve (0.521 plain, 0.481 weighted, 0.498 sampler) because its precision falls from 0.533 to about 0.4.

### 4.3 Image enhancement (hair removal + CLAHE)

Same model, imbalance strategy and split; only the input images change (raw or enhanced). Since the run-to-run noise is around 0.005 (§4.6), a smaller change does not count as a gain, and the simpler raw pipeline is then kept.

In [ ]:
# the same recipe on the enhanced images, kept only if validation macro-F1 improves by at least 0.005
train_model(f"r18_{IMB_BEST}_enh", "resnet18", DATA_ENH, pretrained=True, stages=tl_stages(FT_LR_DEFAULT), amp_dtype=AMP_FP16,
            **IMB_OPTIONS[IMB_BEST])

enh_rows = [summary_row(f"raw images ({IMB_BEST})", evaluate(f"r18_{IMB_BEST}", DATA, "val")[0]),
            summary_row(f"hair removal + CLAHE ({IMB_BEST})", evaluate(f"r18_{IMB_BEST}_enh", DATA_ENH, "val")[0])]
enh_table = pd.DataFrame(enh_rows).set_index("model")
show_table(enh_table.round(3), "Raw against enhanced images (hair removal + CLAHE) on the validation set.")
show_table(per_class_table([f"r18_{IMB_BEST}", f"r18_{IMB_BEST}_enh"], [DATA, DATA_ENH], labels=["raw", "enhanced"]).round(3),
           "Raw against enhanced images: per-class recall, precision and F1 on the validation set.")
gain = enh_table["macro_f1"].iloc[1] - enh_table["macro_f1"].iloc[0]
USE_ENH = gain >= 0.005
DATA_BEST = DATA_ENH if USE_ENH else DATA
print(f"change in validation macro-F1: {gain:+.3f} -> using {'enhanced' if USE_ENH else 'raw'} images from here on")

**Result.** Hair removal + CLAHE lowers validation macro-F1 from 0.724 to 0.666 and melanoma recall from 0.642 to 0.553 (Table 13); balanced accuracy hardly changes (0.743 against 0.746) and AUC drops slightly (0.964 to 0.951). The loss is far beyond the 0.005 threshold, so the raw images are kept. Per class (Table 14), enhancement raises recall for `akiec`, `bcc` and `vasc` but lowers precision in six of the seven classes; the largest F1 losses are `df` (0.778 to 0.549) and `mel` (0.498 to 0.413).

### 4.3b Augmentation test: flips and large rotations

Dermatoscopic images have no fixed orientation (§3.1), but I still test this rather than assume it. Three settings are compared on the same ResNet18 recipe: **full** (the default: flips, rotation up to ±180°, zoom, shift, brightness and contrast), **mild** (no flips, rotation up to ±15°, weaker jitter) and **none**. Another setting replaces the default only if it beats it by at least 0.005 macro-F1.

In [ ]:
# three augmentation presets on the same recipe (the "full" run already exists from §4.2)
AUG_REF = f"r18_{IMB_BEST}_enh" if USE_ENH else f"r18_{IMB_BEST}"
for preset in ("mild", "none"):
    train_model(f"r18_aug_{preset}", "resnet18", DATA_BEST, pretrained=True, stages=tl_stages(FT_LR_DEFAULT),
                amp_dtype=AMP_FP16, aug=preset, **IMB_OPTIONS[IMB_BEST])

aug_names = [AUG_REF, "r18_aug_mild", "r18_aug_none"]
aug_table = pd.DataFrame([summary_row(n, evaluate(n, DATA_BEST, "val")[0]) for n in aug_names]).set_index("model")
aug_table.index = ["full (default)", "mild (no flips, rotation <= 15 deg)", "none"]
show_table(aug_table.round(3), "ResNet18 with three augmentation settings on the validation set.")
plot_histories(aug_names, "ResNet18 with three augmentation settings (validation)")
show_table(per_class_table(aug_names, DATA_BEST, labels=["full", "mild", "none"]).round(3),
           "ResNet18 with three augmentation settings: per-class recall, precision and F1 on the validation set.")

aug_f1 = dict(zip(("full", "mild", "none"), aug_table["macro_f1"]))
best_aug = max(aug_f1, key=aug_f1.get)
DEFAULT_AUG = best_aug if aug_f1[best_aug] - aug_f1["full"] >= 0.005 else "full"      # used by every later run
print("augmentation used from here on:", DEFAULT_AUG)

**Result.** Validation macro-F1 is 0.724 with the full augmentation, 0.714 with mild and 0.692 with none (Table 15), and melanoma recall falls from 0.642 to 0.478 and 0.421. Accuracy moves the other way (0.784, 0.818, 0.819) while balanced accuracy falls (0.743, 0.682, 0.658), so weaker augmentation favours the majority class. Neither weaker setting beats the default, so I keep the full augmentation. Per class (Table 16), dropping flips and large rotations raises `nv` recall (0.825 to 0.921 and 0.943) and lowers recall for `bcc`, `df` and `mel`. Melanoma F1 hardly changes (0.498, 0.490, 0.489) because its precision rises as its recall falls; the largest F1 drops are `bcc` (0.816 to 0.741 and 0.738) and, without augmentation, `df` (0.778 to 0.560).

### 4.4 Hyper-parameter search (random search)

Week 6 ranks the learning rate as the most important hyper-parameter, followed by weight decay and batch size. **Random search** tries more distinct values per hyper-parameter than a grid with the same budget (Bergstra & Bengio, 2012). Each trial is a short ResNet18 run (1 head and 3 fine-tuning epochs, same seed) with the learning rate log-uniform in 3e-5 … 1e-3, weight decay log-uniform in 1e-5 … 1e-2 and batch size 32, 64 or 128. Trial 0 is the default; the best **validation** macro-F1 wins.

In [ ]:
# random search; trial 0 is the default setting
from matplotlib.ticker import NullFormatter

rng = np.random.RandomState(SEED)
trials = [{"lr": float(10 ** rng.uniform(np.log10(3e-5), np.log10(1e-3))),
           "wd": float(10 ** rng.uniform(-5, -2)),
           "bs": int(rng.choice([32, 64, 128]))} for _ in range(N_SEARCH_TRIALS)]
trials[0] = {"lr": FT_LR_DEFAULT, "wd": 1e-4, "bs": BATCH_SIZE}

search_rows = []
for k, hp in enumerate(trials):
    name = f"r18_search{k}"
    train_model(name, "resnet18", DATA_BEST, pretrained=True, stages=tl_stages(hp["lr"], head_epochs=1, ft_epochs=SEARCH_FT_EPOCHS),
                amp_dtype=AMP_FP16, weight_decay=hp["wd"], batch_size=hp["bs"], patience=99, verbose=False, **IMB_OPTIONS[IMB_BEST])
    m = evaluate(name, DATA_BEST, "val")[0]
    search_rows.append({"trial": k, **hp, "val_macro_f1": m["macro_f1"], "val_acc": m["accuracy"], "val_bal_acc": m["balanced_accuracy"]})
search = pd.DataFrame(search_rows).set_index("trial")
show_table(search.round({"lr": 6, "wd": 6, "val_macro_f1": 3, "val_acc": 3, "val_bal_acc": 3}),
           "Random hyper-parameter search: learning rate, weight decay and batch size of every trial with its validation scores.")

BEST_HP = trials[int(search["val_macro_f1"].idxmax())]
print("best hyper-parameters:", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in BEST_HP.items()})

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, col, log in zip(axes, ["lr", "wd", "bs"], [True, True, False]):
    ax.scatter(search[col], search["val_macro_f1"], s=45, c=["#d95f02" if k == 0 else "#4c72b0" for k in search.index])
    for k in search.index:                                              # trial number next to each point
        ax.annotate(str(k), (search.loc[k, col], search.loc[k, "val_macro_f1"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
    ax.set_xlabel({"lr": "learning rate", "wd": "weight decay", "bs": "batch size"}[col])
    ax.set_ylabel("val macro-F1")
    if log:
        ax.set_xscale("log")
        ax.xaxis.set_minor_formatter(NullFormatter())                  # otherwise the tick labels overlap
    else:
        ax.set_xticks([32, 64, 128]); ax.set_xlim(20, 140)             # show all three candidates
show_fig("Validation macro-F1 of every search trial against learning rate (left), weight decay (middle) and batch size (right). The numbers are trial numbers; orange is trial 0, the default.")

search_names = [f"r18_search{k}" for k in range(len(trials))]
show_table(per_class_table(search_names, DATA_BEST, labels=[f"trial {k}" for k in range(len(trials))], metrics=("f1",)).round(3),
           "Random search: per-class F1 of every trial on the validation set.")

**Result.** None of the random trials beats trial 0, the default (validation macro-F1 0.697; the other five range from 0.619 to 0.685), so the default is kept (Table 17). Batch size 128 was never drawn. The worst trial, trial 1, has the highest learning rate (4.6e-04). Per class (Table 18) no trial is best everywhere: trial 0 has the highest F1 only for `df` (0.788), and `df`, with 16 validation images, varies most between trials (0.490 to 0.788).

**Limit of the search.** All three hyper-parameters change at once in only six short trials, so a difference cannot be pinned on any single one; the search only shows that none of the tried settings beats the default. EfficientNet-B0 reuses this learning rate, while ViT-B/16 gets a smaller one.

### 4.5 Final pretrained models

All three models use the two-stage schedule and the chosen imbalance strategy and images. I fine-tune ViT-B/16 with lr 3e-5 (set by hand, not searched) in bfloat16, which has the range of float32 and needs no loss scaling; in the benchmark it was as fast as float16 for this model (Appendix B).

In [ ]:
# final models; ViT gets a smaller learning rate and bfloat16 (see above)
HP_STAGES = tl_stages(BEST_HP["lr"])
kw_final = dict(weight_decay=BEST_HP["wd"], batch_size=BEST_HP["bs"], **IMB_OPTIONS[IMB_BEST])

train_model("r18_final", "resnet18", DATA_BEST, pretrained=True, stages=HP_STAGES, amp_dtype=AMP_FP16, **kw_final)
train_model("effb0_final", "efficientnet_b0", DATA_BEST, pretrained=True, stages=HP_STAGES, amp_dtype=AMP_FP16, **kw_final)
train_model("vit_final", "vit_b_16", DATA_BEST, pretrained=True, stages=tl_stages(3e-5), amp_dtype=AMP_BF16,
            weight_decay=BEST_HP["wd"], batch_size=BEST_HP["bs"], **IMB_OPTIONS[IMB_BEST])

FINALS = ["r18_final", "effb0_final", "vit_final"]
fin_table = pd.DataFrame([summary_row(n, evaluate(n, DATA_BEST if n in FINALS else DATA, "val")[0]) for n in [BASELINE] + FINALS]).set_index("model")
fin_table["params (M)"] = [sum(p.numel() for p in RUNS[n]["model"].parameters()) / 1e6 for n in fin_table.index]
fin_table["train min"] = [RUNS[n]["minutes"] for n in fin_table.index]
show_table(fin_table.round(3), "The baseline and the three fine-tuned models on the validation set, with parameters and training time.")
show_table(per_class_table([BASELINE] + FINALS, [DATA] + [DATA_BEST] * len(FINALS), metrics=("recall", "f1")).round(3),
           "The baseline and the three fine-tuned models: per-class recall and F1 on the validation set.")

**Result.** On validation the pretrained models reach macro-F1 0.717 (EfficientNet-B0), 0.724 (ResNet18) and 0.792 (ViT-B/16), against 0.479 for the baseline. They are much larger (4-86 M parameters instead of 1 M); training times are in Table 19. The baseline is a different, small CNN trained from scratch, so this gain mixes architecture and pretraining; I did not train a ResNet18 from scratch to separate the two. ViT-B/16 beats ResNet18 by 0.068, about 14 seed standard deviations on validation (§4.6), but on the test set the pretrained models cannot be ranked reliably (§5.1). Per class (Table 20), the baseline has the lowest F1 in every class; ViT-B/16 has the highest F1 in five of the seven classes (`akiec`, `bkl`, `df`, `mel`, `nv`) and EfficientNet-B0 in `bcc` and `vasc`.

#### Epochs used (the paper trains for 1,000)

Here every run stops on its own: fine-tuning is capped at 25 epochs, and **early stopping** ends a stage after 5 epochs without improvement in validation macro-F1 (Week 6). The table counts epochs over both stages: how many were run, when validation macro-F1 peaked and when validation loss was lowest.

In [ ]:
# epochs run, epoch of the best validation macro-F1 and epoch of the lowest validation loss
epoch_rows = {}
for n in [BASELINE] + FINALS:
    h = pd.DataFrame(RUNS[n]["history"])
    epoch_rows[n] = {"epochs run": len(h),
                     "best val macro-F1 at epoch": int(h.loc[h["val_macro_f1"].idxmax(), "epoch"]),
                     "lowest val loss at epoch": int(h.loc[h["val_loss"].idxmin(), "epoch"]),
                     "train acc (last epoch)": round(float(h["train_acc"].iloc[-1]), 3),
                     "val acc (last epoch)": round(float(h["val_acc"].iloc[-1]), 3),
                     "seconds / epoch (mean)": round(float(h["seconds"].mean()), 1)}
epoch_table = pd.DataFrame(epoch_rows).T
int_cols = ["epochs run", "best val macro-F1 at epoch", "lowest val loss at epoch"]
epoch_table[int_cols] = epoch_table[int_cols].astype(int)
show_table(epoch_table, "Epochs used by the baseline and the final models (training accuracy is measured on augmented batches; seconds per epoch include the validation pass).")

**Why 1,000 epochs are not needed.**

1. *The pretrained models stop improving early.* Their best validation macro-F1 comes at epoch 10 to 23, and by the last epoch their training accuracy is 96-100 % (Table 21; curves in §5.2). Validation loss is lowest at the same epoch for ResNet18 and much earlier for EfficientNet-B0 (epoch 9) and ViT-B/16 (epoch 4), and rises afterwards, a sign of over-fitting.
2. *The numbers are not comparable.* The 1,000 epochs refer to DiffMIC's diffusion training, and the paper does not say how the final epoch was chosen or whether a validation set was used.
3. *Cost.* At the epoch times in Table 21, 1,000 epochs would take over an hour for ResNet18 and several hours each for EfficientNet-B0 and ViT-B/16.
4. *The baseline is the exception.* Its best epoch (28 of 36) is near the end of its run, although its validation macro-F1 did not improve over the last 8 epochs. It stays 0.24 to 0.31 validation macro-F1 behind the pretrained models; I did not test a longer schedule.

### 4.6 Run-to-run variability (three seeds)

The final ResNet18 configuration is trained with two more seeds (different initial output layer, batch order and augmentation). The spread of the validation scores is the *noise level* for judging the other differences in this notebook.

In [ ]:
# two more seeds of the final ResNet18 recipe
for s in (1, 2):
    train_model(f"r18_final_s{s}", "resnet18", DATA_BEST, pretrained=True, stages=HP_STAGES, amp_dtype=AMP_FP16, seed=SEED + s, **kw_final)

seed_runs = ["r18_final", "r18_final_s1", "r18_final_s2"]
seed_table = pd.DataFrame([summary_row(n, evaluate(n, DATA_BEST, "val")[0]) for n in seed_runs]).set_index("model")
show_table(seed_table.round(3), "The final ResNet18 recipe with three seeds on the validation set.")
show_table(seed_table.agg(["mean", "std"]).round(4), "Mean and standard deviation over the three seeds.")
NOISE = float(seed_table["macro_f1"].std())
seed_f1 = per_class_table(seed_runs, DATA_BEST, labels=[f"seed {SEED + s}" for s in range(3)], metrics=("f1",))
seed_f1.insert(3, ("f1", "std"), seed_f1["f1"].std(axis=1))
show_table(seed_f1.round(3), "The three ResNet18 seeds: per-class F1 on the validation set and its standard deviation.")

# the earlier ablation effects in units of this standard deviation
effects = pd.DataFrame({"change in validation macro-F1": {"hair removal + CLAHE (4.3)": gain,
                                                          "mild augmentation (4.3b)": aug_f1["mild"] - aug_f1["full"],
                                                          "no augmentation (4.3b)": aug_f1["none"] - aug_f1["full"]}})
effects["in seed std"] = effects["change in validation macro-F1"] / NOISE
effects["beyond 2 x seed std"] = effects["in seed std"].abs() > 2
show_table(effects.round(3), "The earlier ablation effects in units of the seed standard deviation of the validation macro-F1.")

**Result.** The three seeds give validation macro-F1 0.722 to 0.731, a standard deviation of 0.0047 (Table 23). Differences smaller than about 0.009 (two standard deviations) are therefore inconclusive, and the 0.005 threshold of §4.3 and §4.3b is about one standard deviation. In these units (Table 25) the enhancement loss (−12.5) and the loss without augmentation (−6.7) are clear, while mild augmentation (−2.1) only just passes. The estimate rests on three runs and on the validation set, where each run's checkpoint was selected; on the test set the same seeds vary far more (Table 33). Per class (Table 24) the noise is also much larger: the F1 standard deviation across seeds is 0.086 for `akiec` and 0.042 for `vasc`, against 0.004 for `bkl` and 0.015 for `nv`, so per-class F1 differences of 0.1 or less in the rare classes can be seed noise.

### 4.7 GAN-based data diversification

A **class-conditional GAN** (Week 4; Goodfellow et al., 2014) learns to draw a lesion of a requested class. Its images go into the **training set only**, topping every class below 822 training images (11.5 % of the training set) up to that number; only `nv` is above it. I kept the design small and stable on purpose:

| Choice | Reason |
|---|---|
| Resolution **64 × 64** | a GAN trained from scratch on about 7,000 images cannot make sharp 224 px images; the output is up-sampled to 224 px, which blurs it (a known limitation) |
| Generator: noise + class embedding → 4×4 map → 4 × (up-sample, conv, batch-norm, ReLU) → tanh | DCGAN family (Radford et al., 2016); up-sampling plus convolution avoids the checkerboard artefacts of transposed convolution (Odena et al., 2016) |
| Discriminator: **projection** conditioning, **spectral normalisation**, hinge loss | spectral normalisation stabilises training (Miyato et al., 2018); the projection discriminator conditions on the class cheaply (Miyato & Koyama, 2018) |
| All 7 classes of the training split, square-root class balancing | the rare classes share low-level structure (skin, borders, colour) with the common ones |
| Moving average of the generator weights; flips on real *and* generated images | smoother samples; the flips augment the discriminator's input |
| **Mode-seeking loss** (weight 1) and **minibatch standard deviation** | both fight *mode collapse*: the mode-seeking term penalises similar images from different noise vectors (Mao et al., 2019), and the minibatch statistic lets the discriminator notice a batch without variety (Karras et al., 2018) |
| Training split only | validation and test images never reach the GAN |

The effect on the classifier is judged against the seed noise from §4.6.

In [ ]:
GAN_RES, Z_DIM = 64, 128
GAN_MS_WEIGHT = 1.0                 # weight of the mode-seeking (diversity) term in the generator loss


class GanGenerator(nn.Module):
    """Noise z + class label -> 64x64 RGB image in [-1, 1]."""

    def __init__(self, z_dim=Z_DIM, ch=64):
        super().__init__()
        self.ch = ch
        self.embed = nn.Embedding(N_CLASSES, z_dim)                 # one learned vector per class
        self.fc = nn.Linear(2 * z_dim, ch * 8 * 4 * 4)
        self.bn0 = nn.BatchNorm2d(ch * 8)
        up = lambda i, o: nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(i, o, 3, padding=1, bias=False),
                                        nn.BatchNorm2d(o), nn.ReLU(True))
        self.body = nn.Sequential(up(ch * 8, ch * 4), up(ch * 4, ch * 2), up(ch * 2, ch), up(ch, ch // 2),
                                  nn.Conv2d(ch // 2, 3, 3, padding=1), nn.Tanh())

    def forward(self, z, y):
        h = self.fc(torch.cat([z, self.embed(y)], 1)).view(-1, self.ch * 8, 4, 4)
        return self.body(F.relu(self.bn0(h)))


class MinibatchStd(nn.Module):
    """Adds the standard deviation across the batch as an extra feature map (Karras et al., 2018),
    so the discriminator can notice a batch of near-identical images."""

    def forward(self, x):
        s = x.float().std(dim=0, unbiased=False).mean()
        return torch.cat([x, s.expand(x.size(0), 1, x.size(2), x.size(3)).to(x.dtype)], 1)


class GanDiscriminator(nn.Module):
    """Image + class label -> realness score (spectral norm, class projection, minibatch std)."""

    def __init__(self, ch=64):
        super().__init__()
        sn = nn.utils.spectral_norm
        self.body = nn.Sequential(sn(nn.Conv2d(3, ch, 4, 2, 1)), nn.LeakyReLU(0.2, True),
                                  sn(nn.Conv2d(ch, ch * 2, 4, 2, 1)), nn.LeakyReLU(0.2, True),
                                  sn(nn.Conv2d(ch * 2, ch * 4, 4, 2, 1)), nn.LeakyReLU(0.2, True),
                                  sn(nn.Conv2d(ch * 4, ch * 8, 4, 2, 1)), nn.LeakyReLU(0.2, True))
        self.mbstd = MinibatchStd()
        self.linear = sn(nn.Linear(ch * 8 + 1, 1))
        self.embed = sn(nn.Embedding(N_CLASSES, ch * 8))

    def forward(self, x, y):
        h = self.mbstd(self.body(x)).sum(dim=(2, 3))                              # [B, ch*8 + 1]
        return self.linear(h).squeeze(1) + (self.embed(y) * h[:, :-1]).sum(1)     # general score + class score


def flip_aug(x):
    """Random flips, applied to real and generated images before the discriminator sees them."""
    b = x.shape[0]
    x = torch.where(torch.rand(b, 1, 1, 1, device=x.device) < 0.5, x.flip(3), x)
    return torch.where(torch.rand(b, 1, 1, 1, device=x.device) < 0.5, x.flip(2), x)


def train_cgan(real_u8, labels, steps, batch=64, seed=SEED):
    """Train the conditional GAN on uint8 images [N, 3, 224, 224] on the device; returns the averaged generator and a log."""
    set_seed(seed)
    real = torch.cat([F.interpolate(real_u8[i:i + 512].float() / 255, size=GAN_RES, mode="area")
                      for i in range(0, len(real_u8), 512)]) * 2 - 1                         # [N, 3, 64, 64] in [-1, 1]
    counts = torch.bincount(labels, minlength=N_CLASSES).clamp(min=1).float()
    weights = (counts ** -0.5)[labels]                                                       # class drawn with probability ~ sqrt(n_c)
    G, D = GanGenerator().to(device), GanDiscriminator().to(device)
    G_avg = copy.deepcopy(G).eval()                                                          # moving average of the generator
    opt_g = torch.optim.Adam(G.parameters(), 1e-4, betas=(0.0, 0.9))
    opt_d = torch.optim.Adam(D.parameters(), 4e-4, betas=(0.0, 0.9))                         # the discriminator learns faster
    log = []
    for step in tqdm(range(1, steps + 1), desc="training cGAN"):
        idx = torch.multinomial(weights, batch, replacement=True)
        xr, y = real[idx], labels[idx]
        z1, z2 = torch.randn(batch, Z_DIM, device=device), torch.randn(batch, Z_DIM, device=device)
        fake = G(z1, y)

        # discriminator, hinge loss: real images should score above +1, generated ones below -1
        loss_d = F.relu(1 - D(flip_aug(xr), y)).mean() + F.relu(1 + D(flip_aug(fake.detach()), y)).mean()
        opt_d.zero_grad(set_to_none=True)
        loss_d.backward()
        opt_d.step()

        # generator: fool the discriminator, and keep images from different noise different (mode seeking, Mao et al., 2019)
        fake2 = G(z2, y)
        loss_adv = -D(flip_aug(fake), y).mean()
        d_img = (fake - fake2).abs().mean(dim=(1, 2, 3))
        d_z = (z1 - z2).abs().mean(dim=1)
        loss_ms = 1.0 / ((d_img / d_z).mean() + 1e-5)
        loss_g = loss_adv + GAN_MS_WEIGHT * loss_ms
        opt_g.zero_grad(set_to_none=True)
        loss_g.backward()
        opt_g.step()

        with torch.no_grad():                                                                # update the averaged generator
            for p_avg, p in zip(G_avg.parameters(), G.parameters()):
                p_avg.mul_(0.999).add_(p, alpha=0.001)
            for b_avg, b in zip(G_avg.buffers(), G.buffers()):
                b_avg.copy_(b)
        if step % max(steps // 20, 1) == 0:
            log.append({"step": step, "loss_d": loss_d.item(), "loss_g": loss_adv.item(), "loss_ms": loss_ms.item(),
                        "diversity": d_img.mean().item()})
    del real
    return G_avg, log


@torch.no_grad()
def generate_images(G_avg, cls, n, seed):
    """n synthetic images of class cls, up-sampled to 224 x 224, as uint8 [n, 3, 224, 224] on the device."""
    g = torch.Generator(device=device).manual_seed(seed)
    out = []
    for i in range(0, n, 256):
        m = min(256, n - i)
        z = torch.randn(m, Z_DIM, device=device, generator=g)
        img = (G_avg(z, torch.full((m,), cls, device=device, dtype=torch.long)) + 1) / 2
        img = F.interpolate(img, size=IMG_SIZE, mode="bicubic", align_corners=False).clamp(0, 1)
        out.append((img * 255).round().to(torch.uint8))
    return torch.cat(out)


# train the GAN on the training split only, or load the generator saved with the same data and settings
gan_file, gan_meta = OUT_DIR / "gan_generator_ema.pt", OUT_DIR / "gan_meta.json"
gan_signature = f"{data_fingerprint(DATA_BEST)}-{GAN_STEPS}-{SEED}-ms{GAN_MS_WEIGHT}-mbstd"
train_idx = SPLITS["lesion"]["train"]
GAN_FROM_CACHE = gan_file.exists() and gan_meta.exists() and json.loads(gan_meta.read_text()).get("signature") == gan_signature
if GAN_FROM_CACHE:
    G_EMA = GanGenerator().to(device)
    G_EMA.load_state_dict(torch.load(gan_file, map_location=device))
    G_EMA.eval()
    print("cGAN generator loaded from", gan_file.name)
else:
    t0 = time.perf_counter()
    train_rows = torch.as_tensor(train_idx, device=device)
    G_EMA, gan_log = train_cgan(DATA_BEST["x"][train_rows], Y_T[train_rows], GAN_STEPS)
    torch.save(G_EMA.state_dict(), gan_file)
    gan_meta.write_text(json.dumps({"signature": gan_signature, "log": gan_log, "minutes": (time.perf_counter() - t0) / 60}))
    print(f"cGAN trained in {(time.perf_counter() - t0) / 60:.1f} min")
GAN_MINUTES = json.loads(gan_meta.read_text())["minutes"]

In [ ]:
# top every class up to 11.5 % of the training set with synthetic images
GAN_TARGET = int(0.115 * len(train_idx))
real_counts = np.bincount(Y[train_idx], minlength=N_CLASSES)
n_gen = np.maximum(GAN_TARGET - real_counts, 0)
print(pd.DataFrame({"real train images": real_counts, "synthetic added": n_gen}, index=CLASSES).T.to_string())

xs_list, ys_list = [], []
for c in range(N_CLASSES):
    if n_gen[c] > 0:
        xs_list.append(generate_images(G_EMA, c, int(n_gen[c]), seed=SEED + c))
        ys_list.append(torch.full((int(n_gen[c]),), c, device=device, dtype=torch.long))
XS, YS = torch.cat(xs_list), torch.cat(ys_list)
print("synthetic training images:", tuple(XS.shape))


# diversity: mean per-pixel standard deviation over the images of one class (a collapsed GAN gives almost 0)
def pixel_diversity(x_u8):
    return (x_u8.float() / 255).std(dim=0).mean().item()


div_rows = {}
for c in range(N_CLASSES):
    if n_gen[c] == 0:
        continue
    real_c = DATA_BEST["x"][torch.as_tensor(train_idx[Y[train_idx] == c][:200], device=device)]
    syn_c = XS[YS == c][:200]
    div_rows[CLASSES[c]] = {"real": pixel_diversity(real_c), "synthetic": pixel_diversity(syn_c)}
div_table = pd.DataFrame(div_rows).T
div_table["ratio"] = div_table["synthetic"] / div_table["real"]
show_table(div_table.round(3), "Diversity (mean per-pixel standard deviation within a class) of real and synthetic images.")

# real (left) and synthetic (right) images; layout="none" because the global autolayout shrinks the tiles
NP = 6
fig, axes = plt.subplots(N_CLASSES, 2 * NP + 1, figsize=(15.5, 9.2), layout="none")      # the middle column is an empty spacer
fig.subplots_adjust(left=0.01, right=0.99, top=0.955, bottom=0.01, wspace=0.05, hspace=0.32)
rng = np.random.RandomState(SEED)
for c in range(N_CLASSES):
    for ax in axes[c]:
        ax.axis("off")
    real_pick = rng.choice(train_idx[Y[train_idx] == c], NP, replace=False)
    syn_pick = np.flatnonzero(YS.cpu().numpy() == c)
    syn_pick = rng.choice(syn_pick, NP, replace=False) if len(syn_pick) >= NP else syn_pick
    for k in range(NP):
        axes[c, k].imshow(DATA_BEST["x"][real_pick[k]].permute(1, 2, 0).cpu())
        if k < len(syn_pick):
            axes[c, NP + 1 + k].imshow(XS[syn_pick[k]].permute(1, 2, 0).cpu())
    if len(syn_pick) == 0:                                   # nv: nothing was generated
        axes[c, NP + 1].text(0.0, 0.5, "no synthetic images needed\n(majority class)", fontsize=9, va="center", transform=axes[c, NP + 1].transAxes)
    axes[c, 0].set_title(f"{CLASSES[c]}: real", loc="left", fontsize=9); axes[c, NP + 1].set_title("synthetic", loc="left", fontsize=9)
show_fig("Real training images (left six of each row) and synthetic images (right six) of every class; the majority class needs none.")

# GAN training curves (logged 20 times during training)
gan_log = pd.DataFrame(json.loads(gan_meta.read_text())["log"])
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(gan_log["step"], gan_log["loss_d"], label="discriminator"); axes[0].plot(gan_log["step"], gan_log["loss_g"], label="generator (adversarial part)")
axes[0].set_xlabel("step"); axes[0].set_title("GAN losses"); axes[0].legend(fontsize=7)
axes[1].plot(gan_log["step"], gan_log["diversity"], color="#d95f02"); axes[1].set_xlabel("step")
axes[1].set_title("mean pixel difference between two images of the same class\nmade from different noise (0 = collapsed)")
show_fig("Training curves of the conditional GAN.")

In [ ]:
# realism check: how a ResNet18 trained on real images classifies the synthetic ones
@torch.no_grad()
def classify_u8(model, x_u8, amp=None):
    model.eval(); preds = []
    for i in range(0, len(x_u8), 256):
        with torch.autocast(device_type=device.type, dtype=amp, enabled=amp is not None):
            preds.append(model(normalise(x_u8[i:i + 256].float() / 255)).argmax(1))
    return torch.cat(preds)


pred_syn = classify_u8(RUNS["r18_final"]["model"], XS, RUNS["r18_final"]["amp"])
recognised = pd.Series({CLASSES[c]: (pred_syn[YS == c] == c).float().mean().item() for c in range(N_CLASSES) if n_gen[c] > 0})
print("share of synthetic images that a real-data ResNet18 assigns to the intended class:")
print(recognised.round(2).to_string())

syn_cm = pd.DataFrame(confusion_matrix(YS.cpu().numpy(), pred_syn.cpu().numpy(), labels=range(N_CLASSES)),
                      index=[f"intended {c}" for c in CLASSES], columns=[f"as {c}" for c in CLASSES])
show_table(syn_cm.loc[syn_cm.sum(axis=1) > 0], "Synthetic images: intended class (rows) against the class a real-data ResNet18 assigns (columns).")

In [ ]:
# ResNet18 on real + synthetic images, compared with the three real-only seeds (validation)
train_model("r18_gan", "resnet18", DATA_BEST, pretrained=True, stages=HP_STAGES, amp_dtype=AMP_FP16, extra=(XS, YS), **kw_final)

gan_names = ["r18_final", "r18_gan"]
plot_histories(gan_names, "ResNet18 trained with and without GAN images (validation)")
gan_table = pd.DataFrame([summary_row(n, evaluate(n, DATA_BEST, "val")[0]) for n in seed_runs + ["r18_gan"]]).set_index("model")
show_table(gan_table.round(3), "ResNet18 trained with and without GAN images on the validation set.")
seed_mean = gan_table.loc[seed_runs, "macro_f1"].mean()
delta = gan_table.loc["r18_gan", "macro_f1"] - seed_mean
print(f"validation macro-F1: {gan_table.loc['r18_gan', 'macro_f1']:.3f} with GAN images, {seed_mean:.3f} for the real-only seeds ({delta:+.3f})")
plot_class_recall(["r18_final", "r18_gan"], DATA_BEST, "val", "Per-class recall (validation): real data only vs real + GAN images")
show_table(per_class_table(["r18_final", "r18_gan"], DATA_BEST, labels=["real only", "real + GAN"]).round(3),
           "ResNet18 with and without GAN images: per-class recall, precision and F1 on the validation set.")

**Findings.**

1. *Realism.* The gallery (Figure 18) shows blurry, up-sampled 64 px lesions with believable colours and shapes but no fine texture such as pigment network. The synthetic images have 72-90 % of the pixel variety of real ones (Table 26), so the generator does not collapse. But a ResNet18 trained on real images assigns only 1-15 % of them to the intended class (61 % for `vasc`) and calls 76 % of all synthetic images naevus (Table 27): the GAN learned what a lesion looks like in general, not what separates the classes.
2. *Effect.* Against the mean of the three real-only seeds, adding the 2,568 images lowers validation macro-F1 by 0.022 (about 5 seed standard deviations), balanced accuracy by 0.037 and melanoma recall by 0.113, and raises accuracy by 0.026 (Table 28). Per class (Table 29), the three classes that received the most synthetic images (`df`, `vasc`, `akiec`) lose recall; the `df` F1 loss (0.778 to 0.606) is far beyond its seed spread (0.031, Table 24), the `akiec` loss (0.648 to 0.563) is not (0.086), and `bkl`, `mel` and `nv` gain a little F1. The decision uses validation data only, and the GAN images are not used further.
3. *Next step:* a higher-resolution generator with adaptive discriminator augmentation (StyleGAN2-ADA) or a diffusion model (§6).

### 4.8 Reproducing the paper's split (leakage experiment)

I train and evaluate the `r18_final` recipe (same hyper-parameters, images and imbalance strategy) again with the **paper-style protocol**: a random 70/30 split of *images*, with 10 % of the training part held out for validation. Only the split changes, but this also makes the training part smaller (6,309 instead of 7,153 images) and puts different images into validation and test, so a difference cannot be put down to leakage alone. The test comparison is in §5.6.

In [ ]:
# the same ResNet18 recipe on the paper's random image split
DATA_PAPER = {"x": DATA_BEST["x"], "y": Y_T, "idx": SPLITS["paper"]}
train_model("r18_paperstyle", "resnet18", DATA_PAPER, pretrained=True, stages=HP_STAGES, amp_dtype=AMP_FP16, **kw_final)
leak = pd.DataFrame([summary_row("lesion-level split (r18_final)", evaluate("r18_final", DATA_BEST, "val")[0]),
                     summary_row("paper-style random split (r18_paperstyle)", evaluate("r18_paperstyle", DATA_PAPER, "val")[0])]).set_index("model")
show_table(leak.round(3), "The same ResNet18 recipe on the lesion-level split and on the paper-style random split (validation sets of each protocol).")
show_table(per_class_table(["r18_final", "r18_paperstyle"], [DATA_BEST, DATA_PAPER], labels=["lesion-level", "paper-style"]).round(3),
           "The same ResNet18 recipe on both split protocols: per-class recall, precision and F1 on each protocol's own validation set.")

**Result (validation sets).** The paper-style split gives accuracy 0.793 and macro-F1 0.688, the lesion-level split 0.784 and 0.724 (Table 30). The two validation sets contain different images (701 against 1,431), and the paper-style one has only 8 images in its rarest class, so one run per protocol cannot separate a leakage effect from chance. Per class (Table 31) the paper-style run is lower on `akiec`, `bcc`, `df` and `vasc` and higher on `bkl`, `mel` and `nv`, but its validation set has only 8 to 36 images in those four classes. §5.6 compares the test sets and tests the leakage directly.

#### Summary of §4 (validation macro-F1, seed standard deviation 0.0047)

| Experiment | Validation macro-F1 | Change | Decision |
|---|---|---|---|
| Transfer learning: ResNet18 against the baseline CNN | 0.479 → 0.724 | +0.245 | adopted |
| Weighted loss against plain loss | 0.670 → 0.719 | +0.049 | tied with the sampler |
| Balanced sampler against plain loss | 0.670 → 0.724 | +0.054 | **adopted** (higher on validation) |
| Hair removal + CLAHE against raw images | 0.724 → 0.666 | −0.058 | rejected |
| Mild augmentation / no augmentation against full | 0.714 / 0.692 | −0.010 / −0.031 (Table 25) | full kept |
| Random search (6 short trials) | best trial = default (0.697) | 0 | default kept |
| EfficientNet-B0 / ViT-B/16 against ResNet18 | 0.717 / 0.792 | −0.007 / +0.068 | all three evaluated on test |
| GAN images against the three-seed mean | 0.725 → 0.703 | −0.022 | rejected |
| Paper-style split against lesion-level split (different validation images) | 0.724 → 0.688 | −0.036 | not comparable, see §5.6 |

## 5. Evaluation, Visualisation and Comparison

From here on the **test set** is used for the final models (the baseline was already tested in §3.3), once per model and only after the model was fixed. I report **accuracy** and **macro-F1** with **95 % bootstrap confidence intervals** (1,000 resamples of the test images), **balanced accuracy**, one-vs-rest **AUC**, **melanoma recall** with its specificity, and per-class sensitivity, specificity, precision and F1 (§5.3).

### 5.1 Test-set results of all final models

In [ ]:
def data_for(name):
    return DATA if name.startswith("cnn") else (DATA_PAPER if name == "r18_paperstyle" else DATA_BEST)


FINAL_NAMES = [BASELINE, "r18_final", "effb0_final", "vit_final", "r18_gan"]
rows, TEST, F1CI = [], {}, {}
for n in FINAL_NAMES:
    m, prob, y, loss = evaluate(n, data_for(n), "test")
    TEST[n] = (m, prob, y)
    pred = prob.argmax(1)
    ci_acc = bootstrap_ci(y, pred, accuracy_score)
    ci_f1 = bootstrap_ci(y, pred, lambda a, b: f1_score(a, b, average="macro", zero_division=0))
    F1CI[n] = ci_f1
    rows.append({"model": n, "accuracy": f"{m['accuracy']:.3f} [{ci_acc[0]:.3f}, {ci_acc[1]:.3f}]",
                 "macro-F1": f"{m['macro_f1']:.3f} [{ci_f1[0]:.3f}, {ci_f1[1]:.3f}]",
                 "balanced acc": round(m["balanced_accuracy"], 3), "AUC": round(m["auc_macro"], 3),
                 "melanoma recall": round(m["mel_recall"], 3), "melanoma specificity": round(m["mel_specificity"], 3),
                 "params (M)": round(sum(p.numel() for p in RUNS[n]["model"].parameters()) / 1e6, 1),
                 "train min": round(RUNS[n]["minutes"], 1)})
test_table = pd.DataFrame(rows).set_index("model")
show_table(test_table, "Test-set results of all final models (95 % bootstrap confidence intervals in brackets), with parameters and training time.")

seed_test = pd.DataFrame([{k: evaluate(n, DATA_BEST, "test")[0][k] for k in ["accuracy", "balanced_accuracy", "macro_f1", "auc_macro", "mel_recall", "mel_specificity"]}
                          for n in seed_runs], index=seed_runs)
show_table(pd.concat([seed_test, pd.DataFrame({"mean": seed_test.mean(), "std": seed_test.std()}).T]).round(3),
           "Test metrics of the three ResNet18 seeds, with their mean and standard deviation (test set).")

val_f1 = {n: evaluate(n, data_for(n), "val")[0]["macro_f1"] for n in FINAL_NAMES}
BEST_NAME = max(val_f1, key=val_f1.get)                      # chosen on validation, not on test
print("best model according to the validation macro-F1:", BEST_NAME)

**Result.**

1. The baseline's macro-F1 interval (0.451 to 0.533) lies below the intervals of all four pretrained models (Table 32), so the gain over the baseline (architecture and pretraining together) is larger than the sampling noise of the test set.
2. The pretrained models cannot be ranked. EfficientNet-B0 has the highest test macro-F1 (0.668), but its interval overlaps with those of the other three, and the model chosen on validation, ViT-B/16, is not re-chosen on test.
3. From validation to test the pretrained models lose 0.085 macro-F1 on average, while the baseline gains 0.013. I made all choices on a validation set whose rarest classes have only 16-20 images, where one image is worth 5-6 points of recall.
4. Melanoma recall differs more between the models than the headline metrics: 0.48 (baseline), 0.67 (ResNet18), 0.72 (EfficientNet-B0), 0.51 (ViT-B/16) and 0.56 (GAN run).
5. Seed noise is much larger on the test set: the three ResNet18 seeds give test macro-F1 0.600 to 0.691 (standard deviation 0.048, against 0.0047 on validation; Table 33), so the differences between the pretrained models (0.620 to 0.668) are within that spread.

### 5.2 Learning curves and efficiency

In [ ]:
# learning curves of the final models, and training time against test macro-F1
plot_histories([BASELINE, "r18_final", "effb0_final", "vit_final", "r18_gan"], "Training and validation curves of all final models", figsize=(16, 4))

fig, ax = plt.subplots(figsize=(6, 4))
for n in FINAL_NAMES:
    ax.scatter(RUNS[n]["minutes"], TEST[n][0]["macro_f1"], s=60)
    ax.annotate(n, (RUNS[n]["minutes"], TEST[n][0]["macro_f1"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("training time (min)"); ax.set_ylabel("test macro-F1"); ax.set_title("Cost-benefit"); show_fig("Cost-benefit: training time against test macro-F1 of every final model.")

**Findings.** As in §4.5, the pretrained models over-fit towards the end of training (training accuracy 0.96-1.00 against 0.77-0.85 on validation), which early stopping handles, while the baseline under-fits (0.588 against 0.648). Figure 23 sets training time against test macro-F1.

### 5.3 Confusion matrices, per-class results and ROC curves

In [ ]:
fig, axes = plt.subplots(1, len(FINAL_NAMES), figsize=(4.0 * len(FINAL_NAMES), 4.2))
for ax, n in zip(axes, FINAL_NAMES):
    m, prob, y = TEST[n]
    plot_confusion(y, prob, f"{n}\nacc {m['accuracy']:.3f}, macro-F1 {m['macro_f1']:.3f}", ax=ax)
show_fig("Row-normalised confusion matrices of all final models on the test set.")

In [ ]:
# one-vs-rest ROC curves of the model chosen on validation
from sklearn.metrics import roc_curve, auc as auc_score

m, prob, y = TEST[BEST_NAME]
fig, ax = plt.subplots(figsize=(5.4, 4.6))
for c in range(N_CLASSES):
    if (y == c).sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(y == c, prob[:, c])
    ax.plot(fpr, tpr, label=f"{CLASSES[c]} (AUC {auc_score(fpr, tpr):.2f})")
ax.plot([0, 1], [0, 1], "k:", lw=1); ax.set_xlabel("false-positive rate"); ax.set_ylabel("true-positive rate")
ax.set_title(f"One-vs-rest ROC curves, {BEST_NAME} (test)"); ax.legend(fontsize=7); show_fig(f"One-vs-rest ROC curves of the best model ({BEST_NAME}) on the test set.")

In [ ]:
# per-class sensitivity (recall), specificity, precision and F1 of every final model on the test set
sens = pd.DataFrame({n: TEST[n][0]["recall"] for n in FINAL_NAMES}, index=CLASSES)
spec = pd.DataFrame({n: TEST[n][0]["specificity"] for n in FINAL_NAMES}, index=CLASSES)
show_table(sens.round(3), "Test-set sensitivity (recall) per class for every final model.")
show_table(spec.round(3), "Test-set specificity per class (one-vs-rest) for every final model.")
mel_ss = pd.DataFrame({n: {"sensitivity": TEST[n][0]["mel_recall"], "specificity": TEST[n][0]["mel_specificity"]} for n in FINAL_NAMES}).T
show_table(mel_ss.round(3), "Melanoma against all other classes: sensitivity and specificity of every final model on the test set.")

prec = pd.DataFrame({n: TEST[n][0]["precision"] for n in FINAL_NAMES}, index=CLASSES)
show_table(prec.round(3), "Test-set precision per class for every final model.")
f1c = pd.DataFrame({n: TEST[n][0]["f1"] for n in FINAL_NAMES}, index=CLASSES)
show_table(f1c.round(3), "Test-set F1 per class for every final model.")

**Findings.**

* Specificity is at least 0.81 for every class and model (Table 35); sensitivity and precision vary much more.
* Every pretrained model recognises `nv` best (recall 0.84-0.93), while the baseline's best class is `vasc` (0.80, against 0.76 for `nv`). The pretrained models confuse `mel`, `nv` and `bkl` most often (Figure 24).
* The weighted baseline spreads its recall evenly (45-48 % for five classes) and finds `akiec` and `vasc` more often than the pretrained models, but with much lower precision (Table 37). In F1 (Table 38) every pretrained model beats the baseline on `bcc`, `bkl`, `df`, `mel` and `nv`, while on `akiec` and `vasc` the baseline is about as good (0.400 and 0.727 against 0.308-0.468 and 0.703-0.788). `akiec` is the weakest class of every pretrained model (recall 0.22-0.39, F1 0.31-0.47), and no model has the best F1 in every class.
* For melanoma, higher sensitivity comes with lower specificity (ViT-B/16 0.509 / 0.953, EfficientNet-B0 0.723 / 0.886); a decision threshold moves along this trade-off (Figure 25). ViT-B/16 has the best accuracy and the highest `nv` recall but finds only half of the melanomas.

**ROC curves (Figure 25).** For ViT-B/16 the one-vs-rest AUC ranges from 0.90 for `mel` (the lowest) to 1.00 for `vasc`, so melanomas are ranked well above chance even where the hard decision misses them; a lower melanoma threshold would trade more false alarms for higher sensitivity.

### 5.4 Grad-CAM: where the network looks

Grad-CAM (Selvaraju et al., 2017) on the last convolutional block checks for *shortcut learning* (a ruler, a vignette or the surrounding skin instead of the lesion). It is applied to the convolutional final model with the best validation macro-F1; ViT has no convolutional feature maps.

In [ ]:
def grad_cam(model, x_norm, target_layer):
    """Grad-CAM maps [B, H, W] in [0, 1] for the predicted class of each image in a normalised batch [B, 3, H, W]."""
    store = {}

    def save_activation(module, inputs, output):          # returning None leaves the layer output unchanged
        store["a"] = output
        output.retain_grad()                              # keep d(score)/d(activation) after backward()

    handle = target_layer.register_forward_hook(save_activation)
    model.eval()
    logits = model(x_norm.clone().requires_grad_(True))
    cls = logits.argmax(1)
    model.zero_grad(set_to_none=True)
    logits.gather(1, cls.view(-1, 1)).sum().backward()
    handle.remove()
    a = store["a"]
    cam = F.relu((a.grad.mean(dim=(2, 3), keepdim=True) * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x_norm.shape[-2:], mode="bilinear", align_corners=False)[:, 0].detach()
    cam = cam - cam.flatten(1).min(1)[0].view(-1, 1, 1)
    return cam / cam.flatten(1).max(1)[0].clamp(min=1e-8).view(-1, 1, 1), cls.detach()


def cam_layer(model, arch):
    return model.layer4 if arch == "resnet18" else model.features[-1]


def overlay(img_u8_chw, cam):
    img = img_u8_chw.permute(1, 2, 0).cpu().numpy()
    heat = cv2.applyColorMap((cam.cpu().numpy() * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1]
    return (0.55 * img + 0.45 * heat).astype(np.uint8)


CNN_FINALS = [n for n in FINAL_NAMES if RUNS[n]["arch"] in ("resnet18", "efficientnet_b0")]
CAM_NAME = max(CNN_FINALS, key=lambda n: val_f1[n])
cam_run, cam_data = RUNS[CAM_NAME], data_for(CAM_NAME)
m, prob, y = TEST[CAM_NAME]
test_idx = cam_data["idx"]["test"]
pred = prob.argmax(1)
print("Grad-CAM model:", CAM_NAME)


def explain(rows_in_test, title):
    """Images (top row) and their Grad-CAM maps (bottom row) for positions in the test split."""
    sel = torch.as_tensor(test_idx[rows_in_test], device=device)
    x_u8 = cam_data["x"][sel]
    cam, cls = grad_cam(cam_run["model"], normalise(x_u8.float() / 255), cam_layer(cam_run["model"], cam_run["arch"]))
    fig, axes = plt.subplots(2, len(rows_in_test), figsize=(2.1 * len(rows_in_test), 4.6), squeeze=False)
    for k, r in enumerate(rows_in_test):
        axes[0, k].imshow(x_u8[k].permute(1, 2, 0).cpu()); axes[0, k].axis("off")
        axes[0, k].set_title(f"true {CLASSES[y[r]]}\npred {CLASSES[pred[r]]} ({prob[r, pred[r]]:.2f})", fontsize=7,
                             color="green" if y[r] == pred[r] else "red")
        axes[1, k].imshow(overlay(x_u8[k], cam[k])); axes[1, k].axis("off")
    fig.suptitle(title, y=1.0); show_fig(title)


# the most confident correct prediction of each class
correct = [int(np.flatnonzero((y == c) & (pred == c))[np.argmax(prob[(y == c) & (pred == c), c])]) for c in range(N_CLASSES)
           if ((y == c) & (pred == c)).any()]
explain(correct, "Correctly classified: the most confident example of each class")

**Findings.** For correct predictions the heat sits on the body and border of the lesion, not on corners, ruler marks or the surrounding skin. The maps cover the whole lesion, so one example per class cannot reveal class-specific structures. Grad-CAM is also coarse (the last feature map is 7 × 7) and shows what the model used, not whether that evidence is clinically valid. Finally, the maps are compact blobs near the centre and the crop puts the lesions in the centre too, so the maps cannot tell "looks at the lesion" apart from a plain centre bias.

### 5.5 Error analysis

In [ ]:
wrong = np.flatnonzero(pred != y)
p_wrong, p_right = prob.max(1)[pred != y], prob.max(1)[pred == y]                  # probability of the predicted class
print(f"mean probability of the predicted class: {p_right.mean():.2f} when correct, {p_wrong.mean():.2f} when wrong")
print(f"probability above 0.9: {(p_wrong > 0.9).sum()} of {len(p_wrong)} wrong predictions ({100 * (p_wrong > 0.9).mean():.0f} %), "
      f"{100 * (p_right > 0.9).mean():.0f} % of the correct ones")
conf_wrong = wrong[np.argsort(-prob[wrong, pred[wrong]])][:7]                     # the most confident mistakes
explain(list(conf_wrong), "The 7 most confident mistakes (Grad-CAM shows the predicted class)")

cm = confusion_matrix(y, pred, labels=range(N_CLASSES))
off = [(CLASSES[i], CLASSES[j], int(cm[i, j])) for i in range(N_CLASSES) for j in range(N_CLASSES) if i != j]
show_table(pd.DataFrame(sorted(off, key=lambda t: -t[2])[:8], columns=["true class", "predicted as", "images"]), "The eight most frequent confusions of the Grad-CAM model on the test set.")

mel_missed = np.flatnonzero((y == MEL) & (pred != MEL))
print(f"melanomas in the test set: {(y == MEL).sum()}, missed: {len(mel_missed)}  (recall {1 - len(mel_missed) / max((y == MEL).sum(), 1):.2f})")
if len(mel_missed):
    print("missed melanomas were predicted as:", pd.Series([CLASSES[p] for p in pred[mel_missed]]).value_counts().to_dict())
    explain(list(mel_missed[:7]), "Missed melanomas")

**Findings.**

* The most common confusions (Table 39) are between classes that are also hard for people: naevus called melanoma (103 of 958 naevi, the main source of false melanoma alarms), melanoma called naevus (32 of 159) or keratosis (16), and keratosis called melanoma (30).
* Wrong predictions are less confident on average than correct ones (0.73 against 0.91), but almost a quarter of them still have a probability above 0.9 (printed above). Without calibration the probability is a warning sign, not a trustworthy confidence.
* The seven most confident mistakes (probability about 1.00) are three low-contrast naevi called `bcc` or `bkl`, three `akiec` lesions (two called `bcc`, one with dark globules called `bkl`) and one clearly pigmented melanoma called `nv`.
* The missed melanomas include small lesions, lesions covered by hair and large amorphous ones. Four of the seven shown were predicted with low confidence (0.37-0.47), the other three with 0.72, 0.94 and 0.94. A lower melanoma threshold would catch some of them at the cost of more false alarms (ROC curves).

### 5.6 Comparison with the baseline and with the paper

In [ ]:
PAPER_BASELINES = {"LDAM": (0.857, 0.734), "OHEM": (0.818, 0.660), "MTL": (0.811, 0.667), "DANIL": (0.825, 0.674),
                   "CL": (0.865, 0.739), "ProCo": (0.887, 0.763), "DiffMIC": (PAPER["accuracy"], PAPER["f1"])}

# the paper-style ResNet18 on its own test split (random 70/30 image split)
mp, prob_p, y_p, _ = evaluate("r18_paperstyle", DATA_PAPER, "test")

own = {n: (TEST[n][0]["accuracy"], TEST[n][0]["macro_f1"]) for n in FINAL_NAMES}
comparison = pd.DataFrame(
    [{"source": "this work", "protocol": "lesion-level split", "model": n, "accuracy": a, "F1 (macro)": f} for n, (a, f) in own.items()] +
    [{"source": "this work", "protocol": "paper-style random split", "model": "r18_paperstyle",
      "accuracy": mp["accuracy"], "F1 (macro)": mp["macro_f1"]}] +
    [{"source": "paper (reported)", "protocol": "random 7:3 split", "model": k, "accuracy": a, "F1 (macro)": f}
     for k, (a, f) in PAPER_BASELINES.items()])
show_table(comparison.round(3), "Test results of this work and the results reported in the paper (the protocols differ).")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
labels = [f"{r.model}\n({'mine' if r.source == 'this work' else 'paper'})" for r in comparison.itertuples()]
colors = ["#1b9e77" if s == "this work" else "#7570b3" for s in comparison["source"]]
for ax, col in zip(axes, ["accuracy", "F1 (macro)"]):
    ax.bar(range(len(comparison)), comparison[col], color=colors)
    ax.set_xticks(range(len(comparison)), labels, rotation=75, fontsize=7); ax.set_ylim(0.4, 1.0); ax.set_title(col)
show_fig("Accuracy and macro-F1 of my models (green) and of the methods compared in the paper (purple).")

# the same ResNet18 recipe on both protocols, with bootstrap intervals
pred_p = prob_p.argmax(1)
macro_f1 = lambda a, b: f1_score(a, b, average="macro", zero_division=0)
r18, y_r18, pred_r18 = TEST["r18_final"][0], TEST["r18_final"][2], TEST["r18_final"][1].argmax(1)
for label, m_, y_, p_, f1_ci in [("lesion-level split", r18, y_r18, pred_r18, F1CI["r18_final"]),
                                 ("paper-style split", mp, y_p, pred_p, bootstrap_ci(y_p, pred_p, macro_f1))]:
    acc_ci = bootstrap_ci(y_, p_, accuracy_score)
    print(f"{label:18s}: accuracy {m_['accuracy']:.3f} [{acc_ci[0]:.3f}, {acc_ci[1]:.3f}], "
          f"macro-F1 {m_['macro_f1']:.3f} [{f1_ci[0]:.3f}, {f1_ci[1]:.3f}]")
print(f"difference (paper-style minus lesion-level): accuracy {mp['accuracy'] - r18['accuracy']:+.3f}, macro-F1 {mp['macro_f1'] - r18['macro_f1']:+.3f}")

# direct leakage test: paper-style test images whose lesion is or is not in its training part
train_lesions_p = set(meta["lesion_id"].iloc[SPLITS["paper"]["train"]])
seen = meta["lesion_id"].iloc[SPLITS["paper"]["test"]].isin(train_lesions_p).to_numpy()
for label, sub in [("lesion seen in training", seen), ("unseen lesion", ~seen)]:
    yt, yp = y_p[sub], pred_p[sub]
    print(f"{label:23s}: {sub.sum():4d} images, {100 * (yt == CLASS_IDX['nv']).mean():.0f} % nv | accuracy {(yp == yt).mean():.3f}, "
          f"balanced accuracy {balanced_accuracy_score(yt, yp):.3f}, macro-F1 {macro_f1(yt, yp):.3f}")
n_seen, n_new = (np.bincount(y_p[s], minlength=N_CLASSES) for s in (seen, ~seen))
rec_seen, rec_new = (np.bincount(y_p[s][pred_p[s] == y_p[s]], minlength=N_CLASSES) / np.maximum(n, 1) for s, n in ((seen, n_seen), (~seen, n_new)))
usable = [c for c in range(N_CLASSES) if n_seen[c] >= 10 and n_new[c] >= 10]          # at least 10 images in both groups
higher = [CLASSES[c] for c in usable if rec_seen[c] > rec_new[c]]
print(f"recall is higher on seen lesions for {len(higher)} of {len(usable)} classes: {', '.join(higher)}")

**Result.** With the same ResNet18 recipe, the paper-style split gives +0.017 accuracy and +0.086 macro-F1 over the lesion-level split (printed above). The accuracy intervals overlap and the macro-F1 intervals do not, but the two protocols use different test images, and three seeds of the lesion-level recipe already span 0.600 to 0.691 test macro-F1, so this difference alone is weak evidence. The direct test is clearer: on the 992 paper-style test images (33 %) whose lesion is also in the training part, the model reaches macro-F1 0.767 and balanced accuracy 0.776, against 0.646 and 0.637 on the 2,013 images of unseen lesions, and recall is higher on the seen lesions for 6 of the 7 classes. Accuracy points the other way (0.722 against 0.822) because naevi are 51 % of the seen-lesion images but 75 % of the unseen ones. The two groups also differ in lesion type, so this is an indication, not a controlled measurement.

*Gap to the paper:* ViT-B/16, the model chosen on validation, is 0.085 accuracy and 0.166 F1 below DiffMIC's reported numbers, and the paper-style ResNet18 0.117 and 0.109 (Table 40). Moving to the leaky split raises the ResNet18 macro-F1 by 0.086, about 44 % of its 0.196 gap to DiffMIC; both numbers come from single runs.

**Baseline against the improved models.** Every pretrained model clearly beats the baseline in accuracy, macro-F1 and balanced accuracy, and none of their macro-F1 intervals overlaps the baseline's (§5.1). Melanoma recall rises from 0.48 to 0.67 with ResNet18 and 0.72 with EfficientNet-B0, but only to 0.51 with ViT-B/16 and 0.56 with the GAN run, so the model with the best accuracy finds the fewest melanomas among the pretrained models. The effect of each §4 step is summarised in the table at the end of §4.

**Comparison with the paper.** The numbers cannot be compared directly:

1. **Different split.** The paper uses a random image-level split, my main results a lesion-level one. Leakage explains part of the gap (Result above), but even with the leaky split the ResNet18 stays well below the reported numbers, so it is not the only difference.
2. **Different effort.** DiffMIC trains for 1,000 epochs with a diffusion head, region patches and a dual-granularity guidance network; my models train for at most a few dozen epochs (§4.5).
3. **Different metric details.** The paper does not say how its F1 is averaged (I use macro, the stricter choice for imbalanced data) and reports neither confidence intervals nor melanoma recall.
4. **Single runs.** The paper gives one number per method. In my runs the seed spread is small for accuracy (standard deviation 0.003 on test) but large for macro-F1 (0.048, Table 33), and the F1 gap between DiffMIC and ProCo (0.053) is of that size.

So this does **not** show that DiffMIC is better or worse than my models; the evidence in the paper is not enough to tell.

## 6. Conclusion and Future Directions

**What was done.** A reproducible pipeline for 7-class skin-lesion classification on HAM10000: a lesion-level split and data checks, a from-scratch CNN baseline and a series of controlled improvements, evaluated with bootstrap confidence intervals, per-class recall, precision and F1, ROC curves and Grad-CAM.

**Main findings** (details in §4 and §5):

1. *The pretrained models gave the largest gain over the baseline* (test macro-F1 0.62-0.67 against 0.49). The baseline differs both in architecture and in being trained from scratch, so the gain comes from both. The pretrained models cannot be ranked reliably: three ResNet18 seeds alone span test macro-F1 0.60 to 0.69. Melanoma recall improves for ResNet18 and EfficientNet-B0 but hardly for ViT-B/16.
2. *Imbalance handling changes which errors are made.* Balanced sampling or class weights raised validation macro-F1 by about 0.05 and melanoma recall from 0.51 to 0.64 for about four points of accuracy, almost as much as switching from ResNet18 to ViT-B/16 (+0.07 on validation). The gain is in the rare classes; melanoma F1 does not improve because its precision falls.
3. *Not every idea helps.* Hair removal + CLAHE (−0.058) and the GAN images (−0.022), which were varied but not class-specific, lowered validation macro-F1; flips and large rotations helped a little (+0.01 to +0.03).
4. *The split protocol matters.* A third of the test images of the paper's random split show a lesion that is also in the training set. With that split the same ResNet18 recipe gains 0.086 test macro-F1 (0.707 against 0.620, inside the seed spread), and recall is higher on leaked than on unseen images for six of the seven classes. Even so, it stays 0.117 accuracy and 0.109 F1 below DiffMIC's reported numbers.
5. *Grad-CAM* puts the evidence on the lesion, with the caveat of a possible centre bias. ResNet18 misses about a third of the melanomas (52 of 159), and 23 % of its errors have a probability above 0.9.

**Limitations.** One dataset and no external validation; labels that are partly expert opinion; a single split; a small search budget (six short trials); no from-scratch run of the pretrained architectures, so pretraining and architecture are not separated; a 64 px GAN whose images are not class-specific; seed noise estimated from three runs of one model (0.005 on validation, 0.048 on test); all tuning on a validation set with only 16-20 images in the rarest classes; no calibration analysis; patient metadata not used.

**Future directions.** Repeated lesion-grouped cross-validation and external validation (ISIC 2019/2020); calibration, ensembles, test-time augmentation and higher input resolution; focal or class-balanced losses and a tuned melanoma threshold; a class-specific generator (StyleGAN2-ADA or a diffusion model); a faithful DiffMIC re-implementation on the lesion-level split; and prospective clinical validation that includes fairness across skin tones.

## References

* Bengio, Y. (2012). Practical recommendations for gradient-based training of deep architectures. In *Neural Networks: Tricks of the Trade*, 2nd ed., Springer, 437–478.
* Bergstra, J. and Bengio, Y. (2012). Random search for hyper-parameter optimization. *JMLR* 13, 281–305.
* Codella, N. et al. (2019). Skin lesion analysis toward melanoma detection 2018: a challenge hosted by the International Skin Imaging Collaboration (ISIC). arXiv:1902.03368.
* Dosovitskiy, A. et al. (2021). An image is worth 16×16 words: Transformers for image recognition at scale. *ICLR*.
* Esteva, A. et al. (2017). Dermatologist-level classification of skin cancer with deep neural networks. *Nature* 542, 115–118.
* Goodfellow, I. et al. (2014). Generative adversarial nets. *NeurIPS*.
* He, K., Zhang, X., Ren, S. and Sun, J. (2016). Deep residual learning for image recognition. *CVPR*.
* Ioffe, S. and Szegedy, C. (2015). Batch normalization: accelerating deep network training by reducing internal covariate shift. *ICML*.
* Karras, T., Aila, T., Laine, S. and Lehtinen, J. (2018). Progressive growing of GANs for improved quality, stability, and variation. *ICLR*.
* Kingma, D. and Ba, J. (2015). Adam: a method for stochastic optimization. *ICLR*.
* Lee, T., Ng, V., Gallagher, R., Coldman, A. and McLean, D. (1997). DullRazor: a software approach to hair removal from images. *Computers in Biology and Medicine* 27(6), 533–543.
* Loshchilov, I. and Hutter, F. (2019). Decoupled weight decay regularization. *ICLR*.
* Mao, Q., Lee, H.-Y., Tseng, H.-Y., Ma, S. and Yang, M.-H. (2019). Mode seeking generative adversarial networks for diverse image synthesis. *CVPR*.
* Masters, D. and Luschi, C. (2018). Revisiting small batch training for deep neural networks. arXiv:1804.07612.
* Micikevicius, P. et al. (2018). Mixed precision training. *ICLR*.
* Miyato, T. and Koyama, M. (2018). cGANs with projection discriminator. *ICLR*.
* Miyato, T., Kataoka, T., Koyama, M. and Yoshida, Y. (2018). Spectral normalization for generative adversarial networks. *ICLR*.
* Odena, A., Dumoulin, V. and Olah, C. (2016). Deconvolution and checkerboard artifacts. *Distill*.
* Radford, A., Metz, L. and Chintala, S. (2016). Unsupervised representation learning with deep convolutional generative adversarial networks. *ICLR*.
* Raghu, M., Zhang, C., Kleinberg, J. and Bengio, S. (2019). Transfusion: understanding transfer learning for medical imaging. *NeurIPS*.
* Selvaraju, R. et al. (2017). Grad-CAM: visual explanations from deep networks via gradient-based localization. *ICCV*.
* Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I. and Salakhutdinov, R. (2014). Dropout: a simple way to prevent neural networks from overfitting. *JMLR* 15, 1929–1958.
* Tan, M. and Le, Q. (2019). EfficientNet: rethinking model scaling for convolutional neural networks. *ICML*.
* Tschandl, P., Rosendahl, C. and Kittler, H. (2018). The HAM10000 dataset, a large collection of multi-source dermatoscopic images of common pigmented skin lesions. *Scientific Data* 5, 180161.
* Yang, Y., Fu, H., Aviles-Rivero, A. I., Schönlieb, C.-B. and Zhu, L. (2023). DiffMIC: dual-guidance diffusion network for medical image classification. *MICCAI 2023*, LNCS, pp. 95–105.
* Yosinski, J., Clune, J., Bengio, Y. and Lipson, H. (2014). How transferable are features in deep neural networks? *NeurIPS* 27.
* Zuiderveld, K. (1994). Contrast limited adaptive histogram equalization. In *Graphics Gems IV*, Academic Press, 474–485.
* PyTorch and torchvision (pretrained ImageNet-1k weights for ResNet18, EfficientNet-B0 and ViT-B/16): https://pytorch.org/vision/stable/models.html
* Course material CM4126: Weeks 1–7 lectures and the Week 6 "Working with Images using TorchVision" notebook.


## Appendix

### A. Windows + AMD GPU workaround

I ran the experiments on an AMD GPU under Windows with AMD's ROCm build of PyTorch. That build has a known bug: BatchNorm layers are passed to MIOpen, which compiles a GPU kernel at run time and fails with *`'type_traits' file not found` → `miopenStatusUnknownError`* (ROCm/TheRock issue #9181), because the wheels ship without the C++ standard-library headers. The cell in §1.1 replaces `torch.nn.functional.batch_norm` with a call to PyTorch's own GPU kernel (the same maths, checked against the CPU during set-up). It also sets `ROCM_SDK_TARGET_FAMILY` to skip a GPU-detection probe that fails on paths containing spaces. On NVIDIA, Linux, macOS and CPU the cell does nothing.

### B. Hardware benchmark that guided the settings

Measured with a separate script, `benchmark_gpu.py`, on the reference machine (Radeon RX 7900 XT, 20 GB), with 224 × 224 inputs and synthetic data on the GPU:

| Model | Best precision | Training speed (images/s) | Speed-up vs fp32 | Estimated epoch (7,010 images) |
|---|---|---|---|---|
| small CNN | fp16 | 1,146 | 1.7× | 6.1 s |
| ResNet18 | fp16 | 1,576 | 2.5× | 4.4 s |
| EfficientNet-B0 | fp16 | 448 | 1.7× | 15.7 s |
| ViT-B/16 | bf16 | 340 | 4.8× | 20.6 s |

* Raw capability: 20 TFLOPS (fp32), 86 TFLOPS (fp16/bf16), 556 GB/s memory bandwidth.
* The estimates assume 7,010 training images (70 % of the data); the lesion-level training split has 7,153. Epochs in the notebook take somewhat longer, because the benchmark times only the training step on synthetic data, while a notebook epoch also includes augmentation, gradient clipping, deterministic kernels and the validation pass.
* Batch sizes of 32–64 already reach the throughput plateau of every model (peak memory below 10 GB), hence batch 64.
* Input pipeline: decoding JPEGs manages about 260 images/s per CPU process. An 8-worker `DataLoader` reaches about 1,500 images/s but is slow to start on Windows. A cache of resized `uint8` images on the GPU with GPU augmentation delivers more than 20,000 images/s, which is why the notebook is built that way.
* bfloat16 was *slower* than float32 for DenseNet-121, and float16 was no faster than bfloat16 for ViT, so the precision is chosen per model.

### C. Training times

In [ ]:
# training time of every experiment and the software versions
runtime = pd.DataFrame({n: {"architecture": r["arch"], "minutes": round(r["minutes"], 1), "loaded from saved model": r["from_cache"]}
                        for n, r in RUNS.items()}).T
runtime.loc["cgan_generator"] = ["conditional GAN", round(GAN_MINUTES, 1), GAN_FROM_CACHE]
show_table(runtime, "Training time of every experiment (minutes recorded when each model was trained).")
print(f"total training time: {sum(r['minutes'] for r in RUNS.values()) + GAN_MINUTES:.1f} min")
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device {device} | seed {SEED}")